# Filtro de Kalman para Estimación de Precios Financieros

## Objetivo del Notebook
Este notebook presenta el **filtro de Kalman** aplicado a la estimación de precios de activos financieros, combinando:
- **Simulación realista** de microestructura de mercado
- **Derivaciones matemáticas** paso a paso del algoritmo de Kalman
- **Implementación práctica** con observaciones duales (mid prices + trades)
- **Visualizaciones interactivas** que ilustran cada concepto

---

## Contenido del Notebook (45 minutos)

1. **📊 Generación de Datos Simulados** (10 min) - Microestructura de mercado con spread variable
2. **🧮 Teoría Matemática** (15 min) - Derivaciones del filtro de Kalman para random walk
3. **⚙️ Implementación** (10 min) - Filtro con ponderación por volumen
4. **📈 Resultados y Análisis** (10 min) - Visualización y evaluación

---

**Audiencia**: Estudiantes de finanzas con conocimientos básicos de probabilidad
**Prerrequisitos**: Conceptos de media, varianza, distribución normal

# 1. Generación de Datos Simulados (Microestructura de Mercado)

En esta sección simularemos datos realistas de microestructura de mercado para un activo financiero durante **300 minutos** (5 horas de trading). Los datos incluirán:

- **Precio verdadero** (no observable): Random walk con volatilidad variable
- **Spread bid-ask**: Variable entre 0.05% - 0.5%, correlacionado con volatilidad
- **Mid prices**: Precio medio entre bid y ask
- **Trades**: Operaciones ejecutadas con volúmenes variables

## Parámetros de Simulación

| Parámetro | Valor | Descripción |
|-----------|-------|-------------|
| **Duración** | 300 ticks | 5 horas de mercado (1 minuto por tick) |
| **Precio inicial** | $100 | Precio de referencia del activo |
| **Spread base** | 0.05% - 0.5% | Rango del spread bid-ask |
| **Volatilidad** | Variable | Se amplía el spread durante alta volatilidad |

## 1.1 Importar Librerías y Configuración

In [1]:
# Importar librerías esenciales
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Configuración para reproducibilidad
np.random.seed(42)

# Parámetros de simulación
SIMULATION_PARAMS = {
    'n_ticks': 300,                    # 300 minutos (5 horas)
    'initial_price': 100.0,            # Precio inicial en $
    'base_volatility': 0.05,           # Volatilidad base (5% por minuto^0.5)
    'volatility_regime_prob': 0.10,    # Probabilidad de cambio de régimen de volatilidad
    'high_vol_multiplier': 5.0,        # Multiplicador para alta volatilidad
    'base_spread_pct': 0.005,         # Spread base (0.5%)
    'max_spread_pct': 0.02,           # Spread máximo (1%)
    'trade_frequency': 0.3,            # Probabilidad de trade por minuto
    'base_volume': 1000,               # Volumen base para trades
    'volume_std': 500                  # Desviación estándar del volumen
}

print("📊 Configuración completada")
print(f"🕐 Simulando {SIMULATION_PARAMS['n_ticks']} minutos de mercado")
print(f"💰 Precio inicial: ${SIMULATION_PARAMS['initial_price']}")
print(f"📈 Spread: {SIMULATION_PARAMS['base_spread_pct']*100:.2f}% - {SIMULATION_PARAMS['max_spread_pct']*100:.1f}%")

📊 Configuración completada
🕐 Simulando 300 minutos de mercado
💰 Precio inicial: $100.0
📈 Spread: 0.50% - 2.0%


## 1.2 Simulación del Precio Verdadero (Random Walk)

El **precio verdadero** del activo (no observable directamente) sigue un random walk con volatilidad variable:

$$p_{t+1} = p_t + \epsilon_t, \quad \epsilon_t \sim N(0, \sigma_t^2)$$

donde $\sigma_t$ puede cambiar entre dos regímenes:
- **Régimen normal**: Volatilidad base
- **Régimen volátil**: Volatilidad amplificada (× 3)

In [2]:
def generate_true_price_path(params):
    """
    Genera la trayectoria del precio verdadero con volatilidad variable.
    
    Returns:
    - true_prices: Array con precios verdaderos
    - volatilities: Array con volatilidades utilizadas
    - vol_regimes: Array binario indicando régimen (0=normal, 1=alto)
    """
    n_ticks = params['n_ticks']
    initial_price = params['initial_price']
    base_vol = params['base_volatility']
    regime_prob = params['volatility_regime_prob']
    high_vol_mult = params['high_vol_multiplier']
    
    # Inicializar arrays
    true_prices = np.zeros(n_ticks)
    volatilities = np.zeros(n_ticks)
    vol_regimes = np.zeros(n_ticks, dtype=int)
    
    # Precio inicial
    true_prices[0] = initial_price
    current_regime = 0  # Empezar en régimen normal
    
    for t in range(1, n_ticks):
        # Cambio de régimen de volatilidad (proceso de Markov simple)
        if np.random.random() < regime_prob:
            current_regime = 1 - current_regime  # Alternar régimen
        
        vol_regimes[t] = current_regime
        
        # Volatilidad actual
        if current_regime == 0:
            volatilities[t] = base_vol
        else:
            volatilities[t] = base_vol * high_vol_mult
        
        # Innovación del precio (random walk)
        price_innovation = np.random.normal(0, volatilities[t])
        true_prices[t] = true_prices[t-1] + price_innovation
    
    return true_prices, volatilities, vol_regimes

# Generar trayectoria del precio verdadero
true_prices, volatilities, vol_regimes = generate_true_price_path(SIMULATION_PARAMS)

print(f"✅ Precio verdadero generado")
print(f"📊 Precio final: ${true_prices[-1]:.2f}")
print(f"📈 Rendimiento total: {((true_prices[-1]/true_prices[0]) - 1)*100:.1f}%")
print(f"🔥 Períodos de alta volatilidad: {vol_regimes.sum()} de {SIMULATION_PARAMS['n_ticks']} ({vol_regimes.mean()*100:.1f}%)")

✅ Precio verdadero generado
📊 Precio final: $107.08
📈 Rendimiento total: 7.1%
🔥 Períodos de alta volatilidad: 164 de 300 (54.7%)


## 1.3 Simulación de Microestructura: Bid, Ask y Spread

El **spread bid-ask** varía dinámicamente basado en la volatilidad del mercado:
- Durante períodos **normales**: Spread mínimo (0.05%)
- Durante **alta volatilidad**: Spread amplificado (hasta 0.5%)

Esto simula el comportamiento real donde los market makers amplían spreads cuando aumenta la incertidumbre.

In [3]:
def generate_bid_ask_data(true_prices, vol_regimes, params):
    """
    Genera bid, ask, spread y mid prices basados en el precio verdadero y régimen de volatilidad.
    VERSIÓN MEJORADA: Mid prices pueden desviarse del precio verdadero independientemente.
    
    Returns:
    - bid_prices: Precios de compra
    - ask_prices: Precios de venta  
    - spreads_pct: Spreads como porcentaje del mid price
    - mid_prices: Precios medios (bid + ask) / 2
    - volumes_bid: Volúmenes disponibles en bid
    - volumes_ask: Volúmenes disponibles en ask
    """
    n_ticks = len(true_prices)
    base_spread = params['base_spread_pct']
    max_spread = params['max_spread_pct']
    
    # Arrays para almacenar resultados
    bid_prices = np.zeros(n_ticks)
    ask_prices = np.zeros(n_ticks)
    spreads_pct = np.zeros(n_ticks)
    mid_prices = np.zeros(n_ticks)
    volumes_bid = np.zeros(n_ticks)
    volumes_ask = np.zeros(n_ticks)
    
    for t in range(n_ticks):
        # Spread dinámico basado en régimen de volatilidad
        if vol_regimes[t] == 0:  # Régimen normal
            spread_pct = base_spread + np.random.uniform(0, base_spread)
        else:  # Régimen de alta volatilidad
            spread_pct = np.random.uniform(base_spread * 2, max_spread)
        
        spreads_pct[t] = spread_pct
        
        # MEJORA: Mid price con ruido independiente del precio verdadero
        # El mercado puede estar "mal posicionado" temporalmente
        independent_noise_pct = 0.002  # ±0.2% de ruido independiente
        mid_price_noise = np.random.normal(0, true_prices[t] * independent_noise_pct)
        mid_prices[t] = true_prices[t] + mid_price_noise
        
        # Calcular bid y ask alrededor del mid price (no del precio verdadero)
        half_spread = spread_pct * mid_prices[t] / 2
        bid_prices[t] = mid_prices[t] - half_spread
        ask_prices[t] = mid_prices[t] + half_spread
        
        # Volúmenes en el libro (distribuidos log-normalmente)
        volumes_bid[t] = np.random.lognormal(np.log(5000), 0.5)
        volumes_ask[t] = np.random.lognormal(np.log(5000), 0.5)
    
    return bid_prices, ask_prices, spreads_pct, mid_prices, volumes_bid, volumes_ask

# Generar datos de microestructura
bid_prices, ask_prices, spreads_pct, mid_prices, volumes_bid, volumes_ask = generate_bid_ask_data(
    true_prices, vol_regimes, SIMULATION_PARAMS
)

print(f"✅ Microestructura generada")
print(f"📊 Spread promedio: {spreads_pct.mean()*100:.3f}%")
print(f"📈 Spread mínimo: {spreads_pct.min()*100:.3f}%")
print(f"📉 Spread máximo: {spreads_pct.max()*100:.3f}%")
print(f"📚 Volumen promedio bid: {volumes_bid.mean():.0f}")
print(f"📗 Volumen promedio ask: {volumes_ask.mean():.0f}")

✅ Microestructura generada
📊 Spread promedio: 1.141%
📈 Spread mínimo: 0.505%
📉 Spread máximo: 1.988%
📚 Volumen promedio bid: 5821
📗 Volumen promedio ask: 5866


## 1.4 Simulación de Trades con Volúmenes Variables

Los **trades** representan transacciones reales ejecutadas. Características importantes:
- **Frecuencia**: Mayor durante períodos de alta volatilidad
- **Precios**: Cerca del mid price pero con ruido de ejecución
- **Volúmenes**: Distribuidos log-normalmente con media variable
- **Información**: Trades grandes contienen más información sobre el precio verdadero

In [4]:
def generate_trades(mid_prices, vol_regimes, params, true_prices):
    """
    Genera trades ejecutados con precios y volúmenes realistas.
    VERSIÓN MEJORADA: Trades siguen más de cerca el precio verdadero, no el mid price.
    
    Returns:
    - trade_times: Índices de tiempo donde ocurren trades
    - trade_prices: Precios de ejecución de los trades
    - trade_volumes: Volúmenes de cada trade
    - trade_sides: Lado del trade (1=buy, -1=sell)
    """
    n_ticks = len(mid_prices)
    base_freq = params['trade_frequency']
    base_volume = params['base_volume']
    volume_std = params['volume_std']
    
    trade_times = []
    trade_prices = []
    trade_volumes = []
    trade_sides = []
    
    for t in range(n_ticks):
        # Frecuencia de trades aumenta durante alta volatilidad
        if vol_regimes[t] == 1:  # Alta volatilidad
            trade_freq = base_freq * 2.0
        else:  # Volatilidad normal
            trade_freq = base_freq
        
        # ¿Ocurre un trade en este tick?
        if np.random.random() < trade_freq:
            trade_times.append(t)
            
            # Lado del trade (aleatorio)
            side = np.random.choice([1, -1])  # 1=buy, -1=sell
            trade_sides.append(side)
            
            # Volumen del trade (determinamos primero para calcular precisión)
            if vol_regimes[t] == 1:
                volume_mean = base_volume * 1.5  # Mayor volumen durante volatilidad
            else:
                volume_mean = base_volume
                
            volume = np.random.lognormal(
                np.log(volume_mean) - 0.5 * (volume_std/volume_mean)**2,  # Ajuste para media correcta
                volume_std / volume_mean
            )
            trade_volumes.append(int(volume))
            
            # MEJORA: Precio basado en precio verdadero, no mid price
            # Trades grandes son más precisos (menor ruido)
            volume_weight = min(volume / (base_volume * 2), 3.0)  # Peso 1x a 3x
            base_trade_noise_pct = 0.0001  # 0.01% base noise
            adjusted_noise_pct = base_trade_noise_pct / np.sqrt(volume_weight)  # Menos ruido para volúmenes grandes
            
            # Ruido de ejecución basado en precio verdadero
            price_noise = np.random.normal(0, true_prices[t] * adjusted_noise_pct)
            
            # Bias mínimo por lado (mucho menor que antes)
            side_bias = side * true_prices[t] * 0.00005  # 0.005% bias por lado
            
            # Precio final: precio verdadero + ruido mínimo + bias mínimo
            trade_price = true_prices[t] + price_noise + side_bias
            trade_prices.append(trade_price)
    
    return np.array(trade_times), np.array(trade_prices), np.array(trade_volumes), np.array(trade_sides)

# Generar trades
trade_times, trade_prices, trade_volumes, trade_sides = generate_trades(
    mid_prices, vol_regimes, SIMULATION_PARAMS, true_prices
)

print(f"✅ Trades generados")
print(f"📊 Total de trades: {len(trade_times)}")
print(f"📈 Trades por minuto promedio: {len(trade_times)/SIMULATION_PARAMS['n_ticks']:.2f}")
print(f"💰 Volumen promedio por trade: {trade_volumes.mean():.0f}")
print(f"📊 Volumen total: {trade_volumes.sum():,.0f}")
print(f"⚖️ Ratio buy/sell: {(trade_sides == 1).sum()}/{(trade_sides == -1).sum()}")

✅ Trades generados
📊 Total de trades: 148
📈 Trades por minuto promedio: 0.49
💰 Volumen promedio por trade: 1350
📊 Volumen total: 199,780
⚖️ Ratio buy/sell: 74/74


## 1.5 Organización de Datos en DataFrame

Creamos un DataFrame de pandas para organizar todos los datos simulados y facilitar el análisis posterior.

In [5]:
# Crear DataFrame principal con datos de cada tick
market_data = pd.DataFrame({
    'time': range(SIMULATION_PARAMS['n_ticks']),
    'true_price': true_prices,
    'mid_price': mid_prices,
    'bid_price': bid_prices,
    'ask_price': ask_prices,
    'spread_pct': spreads_pct,
    'spread_bps': spreads_pct * 10000,  # En basis points
    'volatility': volatilities,
    'vol_regime': vol_regimes,
    'volume_bid': volumes_bid,
    'volume_ask': volumes_ask
})

# Crear DataFrame de trades
trades_data = pd.DataFrame({
    'time': trade_times,
    'price': trade_prices,
    'volume': trade_volumes,
    'side': trade_sides,
    'side_str': ['buy' if s == 1 else 'sell' for s in trade_sides]
})

print("✅ DataFrames creados")
print(f"📊 Market data shape: {market_data.shape}")
print(f"💱 Trades data shape: {trades_data.shape}")
print("\n📋 Primeras 5 filas del market data:")
print(market_data.head())

✅ DataFrames creados
📊 Market data shape: (300, 11)
💱 Trades data shape: (148, 5)

📋 Primeras 5 filas del market data:
   time  true_price   mid_price  bid_price   ask_price  spread_pct  \
0     0  100.000000  100.053549  99.683065  100.424032    0.007406   
1     1   99.944406   99.824079  99.490516  100.157642    0.006683   
2     2   99.960351  100.114618  99.715251  100.513984    0.007978   
3     3  100.030111  100.064068  99.389201  100.738934    0.013489   
4     4  100.282740  100.599661  99.673224  101.526097    0.018418   

   spread_bps  volatility  vol_regime   volume_bid   volume_ask  
0   74.057005        0.00           0  3609.321695  5119.911983  
1   66.830214        0.05           0  7312.340934  8693.733379  
2   79.781937        0.05           0  1452.978609  3356.807202  
3  134.886827        0.25           1  5228.554596  6381.862305  
4  184.182878        0.25           1  5919.424572  4069.415712  


## 1.6 Visualización de los Datos Simulados

Creamos gráficos interactivos para entender las características de los datos simulados antes de aplicar el filtro de Kalman.

In [22]:
# Crear visualización principal de precios y spread con tema oscuro
fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=('Precio Verdadero vs Mid Price vs Bid/Ask vs Trades', 'Spread Bid-Ask (basis points)', 'Régimen de Volatilidad', 'Volumen por Trade'),
    vertical_spacing=0.06,
    row_heights=[0.4, 0.2, 0.2, 0.2],
    specs=[[{"secondary_y": False}], 
           [{"secondary_y": False}], 
           [{"secondary_y": False}], 
           [{"secondary_y": False}]]
)

# Gráfico 1: Precios con bid/ask
fig.add_trace(
    go.Scatter(x=market_data['time'], y=market_data['true_price'], 
               name='Precio Verdadero', line=dict(color='white', width=2)),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=market_data['time'], y=market_data['mid_price'], 
               name='Mid Price', line=dict(color='cyan', width=1.5)),
    row=1, col=1
)

# Añadir bid y ask como área sombreada
fig.add_trace(
    go.Scatter(x=market_data['time'], y=market_data['ask_price'], 
               name='Ask Price', line=dict(color='rgba(255,100,100,0.8)', width=1),
               fill=None, mode='lines'),
    row=1, col=1
)

fig.add_trace(
    go.Scatter(x=market_data['time'], y=market_data['bid_price'], 
               name='Bid Price', line=dict(color='rgba(100,255,100,0.8)', width=1),
               fill='tonexty', fillcolor='rgba(150,150,150,0.2)', mode='lines'),
    row=1, col=1
)

# Agregar trades como puntos
if len(trades_data) > 0:
    # Separar buys y sells para diferente color
    buys = trades_data[trades_data['side'] == 1]
    sells = trades_data[trades_data['side'] == -1]
    
    if len(buys) > 0:
        fig.add_trace(
            go.Scatter(x=buys['time'], y=buys['price'], mode='markers',
                       name='Buy Trades', marker=dict(color='lime', size=6, symbol='triangle-up')),
            row=1, col=1
        )
    
    if len(sells) > 0:
        fig.add_trace(
            go.Scatter(x=sells['time'], y=sells['price'], mode='markers',
                       name='Sell Trades', marker=dict(color='red', size=6, symbol='triangle-down')),
            row=1, col=1
        )

# Gráfico 2: Spread
fig.add_trace(
    go.Scatter(x=market_data['time'], y=market_data['spread_bps'], 
               name='Spread (bps)', line=dict(color='orange', width=1), 
               fill='tozeroy', fillcolor='rgba(255,165,0,0.3)'),
    row=2, col=1
)

# Gráfico 3: Régimen de volatilidad
fig.add_trace(
    go.Scatter(x=market_data['time'], y=market_data['vol_regime'], 
               name='Volatilidad Alta', line=dict(color='red', width=1), 
               mode='lines', fill='tozeroy', fillcolor='rgba(255,0,0,0.3)'),
    row=3, col=1
)

# Gráfico 4: Volumen por trade - REDISEÑADO SIN COLORBAR
if len(trades_data) > 0:
    # Separar trades por lado para usar colores fijos sin colorbar
    buys_vol = trades_data[trades_data['side'] == 1]
    sells_vol = trades_data[trades_data['side'] == -1]
    
    # Buy trades en verde
    if len(buys_vol) > 0:
        fig.add_trace(
            go.Scatter(x=buys_vol['time'], y=buys_vol['volume'], 
                       mode='markers',
                       marker=dict(color='lime', size=8, line=dict(width=1, color='white')),
                       name='Volumen Buy',
                       text=[f"Vol: {v:,.0f}<br>Side: Buy" for v in buys_vol['volume']],
                       hovertemplate='<b>Tiempo:</b> %{x}<br><b>Volumen:</b> %{y:,.0f}<br>%{text}<extra></extra>'),
            row=4, col=1
        )
    
    # Sell trades en rojo
    if len(sells_vol) > 0:
        fig.add_trace(
            go.Scatter(x=sells_vol['time'], y=sells_vol['volume'], 
                       mode='markers',
                       marker=dict(color='red', size=8, line=dict(width=1, color='white')),
                       name='Volumen Sell',
                       text=[f"Vol: {v:,.0f}<br>Side: Sell" for v in sells_vol['volume']],
                       hovertemplate='<b>Tiempo:</b> %{x}<br><b>Volumen:</b> %{y:,.0f}<br>%{text}<extra></extra>'),
            row=4, col=1
        )

# Configuración del layout con tema oscuro y leyenda mejorada
fig.update_layout(
    title={
        'text': '📊 Datos Simulados de Microestructura de Mercado',
        'font': {'color': 'white', 'size': 20}
    },
    height=900,
    showlegend=True,
    legend=dict(
        x=1.02, y=1,
        font=dict(color='white', size=11),
        bgcolor='rgba(17, 17, 17, 0.8)',
        bordercolor='rgba(255, 255, 255, 0.2)',
        borderwidth=1
    ),
    plot_bgcolor='rgb(17, 17, 17)',
    paper_bgcolor='rgb(17, 17, 17)',
    font=dict(color='white'),
    # Forzar alineación izquierda de todos los subplots
    margin=dict(l=80, r=150, t=80, b=60)
)

# Actualizar ejes con tema oscuro y forzar alineación izquierda consistente
for i in range(1, 5):
    fig.update_xaxes(
        gridcolor='rgb(68, 68, 68)',
        tickfont=dict(color='white'),
        row=i, col=1
    )
    fig.update_yaxes(
        gridcolor='rgb(68, 68, 68)',
        tickfont=dict(color='white'),
        row=i, col=1
    )

# Forzar alineación izquierda específica para todos los subplots
fig.update_layout(
    xaxis=dict(domain=[0.0, 0.85]),   # Subplot 1
    xaxis2=dict(domain=[0.0, 0.85]),  # Subplot 2  
    xaxis3=dict(domain=[0.0, 0.85]),  # Subplot 3
    xaxis4=dict(domain=[0.0, 0.85])   # Subplot 4
)

# Configurar títulos de ejes con color blanco
fig.update_xaxes(title_text="Tiempo (minutos)", title_font=dict(color='white'), row=4, col=1)
fig.update_yaxes(title_text="Precio ($)", title_font=dict(color='white'), row=1, col=1)
fig.update_yaxes(title_text="Spread (bps)", title_font=dict(color='white'), row=2, col=1)
fig.update_yaxes(title_text="Régimen", title_font=dict(color='white'), row=3, col=1)
fig.update_yaxes(title_text="Volumen", title_font=dict(color='white'), row=4, col=1)

# Actualizar títulos de subplots con color blanco
annotations = list(fig.layout.annotations)
for annotation in annotations:
    annotation.font.color = 'white'
fig.layout.annotations = annotations

fig.show()

print(f"🎯 Visualización completada con tema oscuro y leyenda mejorada")
print(f"📊 Observa cómo el spread (área sombreada entre bid-ask) se amplía durante períodos de alta volatilidad")
print(f"💱 Los trades están distribuidos a lo largo del tiempo con volúmenes variables")
print(f"🔍 El mid price sigue al precio verdadero pero con ruido")
print(f"📈 Volúmenes Buy (verde) y Sell (rojo) claramente diferenciados sin barra vertical")
print(f"✨ Leyenda limpia sin elementos superpuestos")

🎯 Visualización completada con tema oscuro y leyenda mejorada
📊 Observa cómo el spread (área sombreada entre bid-ask) se amplía durante períodos de alta volatilidad
💱 Los trades están distribuidos a lo largo del tiempo con volúmenes variables
🔍 El mid price sigue al precio verdadero pero con ruido
📈 Volúmenes Buy (verde) y Sell (rojo) claramente diferenciados sin barra vertical
✨ Leyenda limpia sin elementos superpuestos


In [7]:
# Análisis adicional: Distribución de volúmenes de trades con tema oscuro
if len(trades_data) > 0:
    fig_volume = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Distribución de Volúmenes de Trades', 'Análisis Temporal de Trades'),
    )
    
    # Histograma de volúmenes
    fig_volume.add_trace(
        go.Histogram(x=trades_data['volume'], nbinsx=30, name='Volúmenes', 
                     marker_color='lightblue', marker_line=dict(color='white', width=1)),
        row=1, col=1
    )
    
    # Scatter plot mejorado: Precio vs Volumen con información temporal
    fig_volume.add_trace(
        go.Scatter(x=trades_data['price'], y=trades_data['volume'], 
                   mode='markers',
                   marker=dict(
                       color=trades_data['time'], 
                       colorscale='Viridis',
                       size=8,
                       line=dict(width=1, color='white'),
                       colorbar=dict(
                           title=dict(text="Tiempo<br>(minutos)", font=dict(color='white')),
                           tickfont=dict(color='white')
                       )
                   ),
                   text=[f"Tiempo: {t}<br>Precio: ${p:.2f}<br>Vol: {v:,.0f}<br>Side: {'Buy' if s==1 else 'Sell'}" 
                         for t, p, v, s in zip(trades_data['time'], trades_data['price'], 
                                             trades_data['volume'], trades_data['side'])],
                   hovertemplate='%{text}<extra></extra>',
                   name='Precio vs Volumen'),
        row=1, col=2
    )
    
    # Configuración del layout con tema oscuro
    fig_volume.update_layout(
        title={
            'text': '📈 Análisis de Volúmenes de Trading',
            'font': {'color': 'white', 'size': 18}
        },
        height=450,
        plot_bgcolor='rgb(17, 17, 17)',
        paper_bgcolor='rgb(17, 17, 17)',
        font=dict(color='white'),
        showlegend=False
    )
    
    # Actualizar títulos de subplots con color blanco
    annotations = list(fig_volume.layout.annotations)
    for annotation in annotations:
        annotation.font.color = 'white'
    fig_volume.layout.annotations = annotations
    
    # Actualizar ejes con tema oscuro
    for i in range(1, 3):
        fig_volume.update_xaxes(
            gridcolor='rgb(68, 68, 68)',
            tickfont=dict(color='white'),
            row=1, col=i
        )
        fig_volume.update_yaxes(
            gridcolor='rgb(68, 68, 68)',
            tickfont=dict(color='white'),
            row=1, col=i
        )
    
    # Configurar títulos de ejes con color blanco
    fig_volume.update_xaxes(title_text="Volumen", title_font=dict(color='white'), row=1, col=1)
    fig_volume.update_xaxes(title_text="Precio ($)", title_font=dict(color='white'), row=1, col=2)
    fig_volume.update_yaxes(title_text="Frecuencia", title_font=dict(color='white'), row=1, col=1)
    fig_volume.update_yaxes(title_text="Volumen", title_font=dict(color='white'), row=1, col=2)
    
    fig_volume.show()

# Estadísticas adicionales sobre los trades
if len(trades_data) > 0:
    print("📊 Estadísticas de Trading:")
    print(f"   • Total trades: {len(trades_data):,}")
    print(f"   • Volumen promedio: {trades_data['volume'].mean():,.0f}")
    print(f"   • Volumen mediano: {trades_data['volume'].median():,.0f}")
    print(f"   • Volumen máximo: {trades_data['volume'].max():,.0f}")
    print(f"   • Volumen mínimo: {trades_data['volume'].min():,.0f}")
    print(f"   • Ratio Buy/Sell: {(trades_data['side']==1).sum()}/{(trades_data['side']==-1).sum()}")
    print(f"   • Precio promedio trades: ${trades_data['price'].mean():.2f}")

print("✅ Sección 1 completada: Datos de microestructura generados")
print("🎯 Los datos simulados incluyen:")
print("   • Precio verdadero (random walk con volatilidad variable)")
print("   • Bid/Ask con spread dinámico (0.05% - 0.5%) - VISIBLE en gráfico")
print("   • Mid prices (observables con ruido = spread/2)")
print("   • Trades con volúmenes variables (información ∝ 1/volumen) - VOLUMEN VISIBLE")
print("   • Correlación entre volatilidad y spread")
print("   • Tema oscuro aplicado a todas las visualizaciones")
print("\n📚 Próximo paso: Teoría matemática del filtro de Kalman")

📊 Estadísticas de Trading:
   • Total trades: 148
   • Volumen promedio: 1,350
   • Volumen mediano: 1,349
   • Volumen máximo: 3,119
   • Volumen mínimo: 403
   • Ratio Buy/Sell: 74/74
   • Precio promedio trades: $105.34
✅ Sección 1 completada: Datos de microestructura generados
🎯 Los datos simulados incluyen:
   • Precio verdadero (random walk con volatilidad variable)
   • Bid/Ask con spread dinámico (0.05% - 0.5%) - VISIBLE en gráfico
   • Mid prices (observables con ruido = spread/2)
   • Trades con volúmenes variables (información ∝ 1/volumen) - VOLUMEN VISIBLE
   • Correlación entre volatilidad y spread
   • Tema oscuro aplicado a todas las visualizaciones

📚 Próximo paso: Teoría matemática del filtro de Kalman


## 🤔 **Pregunta Conceptual Importante: ¿Precio Verdadero vs Mid Price?**

### **¿Qué es el "Precio Verdadero" (True Price)?**

El **precio verdadero** es un **concepto teórico** que representa el "valor fundamental" del activo en ausencia de fricciones de mercado. Es el precio al que se realizarían todas las transacciones si no existieran:

- ❌ Costos de transacción
- ❌ Spreads bid-ask  
- ❌ Efectos de inventario de los dealers
- ❌ Asimetrías de información
- ❌ Restricciones de liquidez

**En la realidad, este precio NO es observable directamente.** Es una variable latente (oculta) que queremos estimar.

### **¿Qué es el Mid Price?**

El **mid price** es una **observación real y calculable**:

$$\text{Mid Price} = \frac{\text{Bid} + \text{Ask}}{2}$$

Es observable en el mercado, pero es una **estimación ruidosa** del precio verdadero porque:
- ✅ Es calculable directamente de las cotizaciones
- ❌ Está afectado por el spread (ruido de microestructura)
- ❌ Durante spreads amplios, se aleja más del precio verdadero

In [8]:
# Demostración práctica de la diferencia conceptual
print("🔍 ANÁLISIS: Precio Verdadero vs Mid Price vs Trades")
print("=" * 60)

# Estadísticas del precio verdadero
print(f"📊 PRECIO VERDADERO (variable latente - NO observable):")
print(f"   • Precio inicial: ${true_prices[0]:.2f}")
print(f"   • Precio final: ${true_prices[-1]:.2f}")
print(f"   • Volatilidad promedio: {volatilities.mean():.4f}")
print(f"   • Desviación estándar: {true_prices.std():.2f}")

print(f"\n📈 MID PRICE (observable pero ruidoso):")
print(f"   • Precio inicial: ${mid_prices[0]:.2f}")
print(f"   • Precio final: ${mid_prices[-1]:.2f}")
print(f"   • Desviación estándar: {mid_prices.std():.2f}")

# Diferencia entre precio verdadero y mid price
price_diff = mid_prices - true_prices
print(f"\n🎯 DIFERENCIA (Mid Price - Precio Verdadero):")
print(f"   • Error promedio: ${price_diff.mean():.4f}")
print(f"   • Error absoluto promedio: ${np.abs(price_diff).mean():.4f}")
print(f"   • Error máximo: ${np.abs(price_diff).max():.4f}")
print(f"   • Correlación: {np.corrcoef(true_prices, mid_prices)[0,1]:.4f}")

# Análisis por régimen de volatilidad
normal_periods = vol_regimes == 0
high_vol_periods = vol_regimes == 1

if high_vol_periods.sum() > 0:
    print(f"\n📊 ERROR POR RÉGIMEN DE VOLATILIDAD:")
    print(f"   • Error en períodos normales: ${np.abs(price_diff[normal_periods]).mean():.4f}")
    print(f"   • Error en alta volatilidad: ${np.abs(price_diff[high_vol_periods]).mean():.4f}")
    print(f"   • Spread promedio normal: {spreads_pct[normal_periods].mean()*100:.3f}%")
    print(f"   • Spread promedio alto vol: {spreads_pct[high_vol_periods].mean()*100:.3f}%")

print(f"\n💡 CONCLUSIÓN:")
print(f"   El mid price es una aproximación al precio verdadero,")
print(f"   pero con error que aumenta durante períodos volátiles.")
print(f"   🎯 ¡Aquí es donde el filtro de Kalman puede ayudarnos!")

🔍 ANÁLISIS: Precio Verdadero vs Mid Price vs Trades
📊 PRECIO VERDADERO (variable latente - NO observable):
   • Precio inicial: $100.00
   • Precio final: $107.08
   • Volatilidad promedio: 0.1592
   • Desviación estándar: 2.95

📈 MID PRICE (observable pero ruidoso):
   • Precio inicial: $100.05
   • Precio final: $107.06
   • Desviación estándar: 2.95

🎯 DIFERENCIA (Mid Price - Precio Verdadero):
   • Error promedio: $0.0011
   • Error absoluto promedio: $0.1561
   • Error máximo: $0.7158
   • Correlación: 0.9978

📊 ERROR POR RÉGIMEN DE VOLATILIDAD:
   • Error en períodos normales: $0.1561
   • Error en alta volatilidad: $0.1560
   • Spread promedio normal: 0.742%
   • Spread promedio alto vol: 1.471%

💡 CONCLUSIÓN:
   El mid price es una aproximación al precio verdadero,
   pero con error que aumenta durante períodos volátiles.
   🎯 ¡Aquí es donde el filtro de Kalman puede ayudarnos!


---

# 2. Teoría Matemática del Filtro de Kalman (15 minutos)

En esta sección desarrollaremos **paso a paso** las ecuaciones del filtro de Kalman para nuestro problema específico de estimación de precios financieros.

## 🎯 **Objetivo**: 
Estimar el **precio verdadero** $p_t$ (no observable) a partir de **observaciones ruidosas**:
- **Mid prices**: $y_t^{mid}$ (siempre disponibles)
- **Trade prices**: $y_t^{trade}$ (esporádicas, con información proporcional al volumen)

## 📐 **Modelo de Espacio de Estados**

### **Ecuación de Estado** (Evolución del Precio Verdadero)
$$p_{t+1} = p_t + w_t, \quad w_t \sim N(0, Q)$$

**Interpretación**: El precio verdadero sigue un **random walk** con varianza $Q$ (volatilidad del mercado).

### **Ecuaciones de Observación** (Mediciones Ruidosas)

**Mid Price**:
$$y_t^{mid} = p_t + v_t^{mid}, \quad v_t^{mid} \sim N(0, R^{mid})$$

**Trade Price** (cuando disponible):
$$y_t^{trade} = p_t + v_t^{trade}, \quad v_t^{trade} \sim N(0, R^{trade})$$

donde:
- $Q$: Varianza del proceso (volatilidad del precio verdadero)
- $R^{mid}$: Varianza del ruido en mid prices (relacionada con spread)
- $R^{trade}$: Varianza del ruido en trades (función del volumen)

## 2.1 Parámetros del Modelo y Notación

Antes de derivar las ecuaciones, definamos claramente todos los elementos del filtro de Kalman:

In [9]:
# Definir parámetros del modelo Kalman basados en nuestros datos simulados
print("🔧 PARÁMETROS DEL FILTRO DE KALMAN")
print("=" * 50)

# Estimar volatilidad del proceso (Q) a partir de los datos simulados
returns = np.diff(true_prices)
Q_estimated = np.var(returns)  # Varianza del random walk
print(f"📊 Varianza del proceso (Q): {Q_estimated:.6f}")
print(f"   → Volatilidad del precio verdadero: {np.sqrt(Q_estimated):.4f}")

# Estimar ruido de observación para mid prices (R_mid)
mid_noise = mid_prices - true_prices
R_mid_estimated = np.var(mid_noise)
print(f"\n📈 Varianza ruido mid prices (R_mid): {R_mid_estimated:.6f}")
print(f"   → Desviación estándar: {np.sqrt(R_mid_estimated):.4f}")

# Estimar ruido de observación para trades (R_trade)
if len(trades_data) > 0:
    # Obtener precios verdaderos en momentos de trades
    true_at_trades = true_prices[trade_times]
    trade_noise = trade_prices - true_at_trades
    R_trade_estimated = np.var(trade_noise)
    print(f"\n💱 Varianza ruido trades (R_trade): {R_trade_estimated:.6f}")
    print(f"   → Desviación estándar: {np.sqrt(R_trade_estimated):.4f}")
else:
    R_trade_estimated = R_mid_estimated * 0.5  # Asumir que trades son más precisos

# Parámetros del filtro
KALMAN_PARAMS = {
    'Q': Q_estimated,                    # Varianza del proceso
    'R_mid': R_mid_estimated,            # Varianza ruido mid prices  
    'R_trade_base': R_trade_estimated,   # Varianza base trades
    'initial_price': true_prices[0],     # Estimación inicial
    'initial_variance': Q_estimated * 10 # Incertidumbre inicial alta
}

print(f"\n📋 RESUMEN DE PARÁMETROS:")
for param, value in KALMAN_PARAMS.items():
    print(f"   • {param}: {value:.6f}")

print(f"\n💡 INTERPRETACIÓN:")
print(f"   • Q grande → Precio verdadero cambia mucho (alta volatilidad)")
print(f"   • R_mid grande → Mid prices muy ruidosos (spreads amplios)")
print(f"   • R_trade pequeño → Trades más informativos que mid prices")
print(f"   • Ratio señal/ruido: {Q_estimated/R_mid_estimated:.2f}")

🔧 PARÁMETROS DEL FILTRO DE KALMAN
📊 Varianza del proceso (Q): 0.037168
   → Volatilidad del precio verdadero: 0.1928

📈 Varianza ruido mid prices (R_mid): 0.038578
   → Desviación estándar: 0.1964

💱 Varianza ruido trades (R_trade): 0.000300
   → Desviación estándar: 0.0173

📋 RESUMEN DE PARÁMETROS:
   • Q: 0.037168
   • R_mid: 0.038578
   • R_trade_base: 0.000300
   • initial_price: 100.000000
   • initial_variance: 0.371675

💡 INTERPRETACIÓN:
   • Q grande → Precio verdadero cambia mucho (alta volatilidad)
   • R_mid grande → Mid prices muy ruidosos (spreads amplios)
   • R_trade pequeño → Trades más informativos que mid prices
   • Ratio señal/ruido: 0.96


## 2.2 Derivación Paso a Paso: Ecuaciones de Predicción

El filtro de Kalman tiene **dos fases**:
1. **Predicción** (a priori): Usar el modelo para predecir el siguiente estado
2. **Corrección** (a posteriori): Usar nuevas observaciones para corregir la predicción

### **Fase 1: Predicción**

**Dado**: Estimación previa $\hat{p}_{t-1|t-1}$ con varianza $P_{t-1|t-1}$

**Predicción del estado**:
$$\hat{p}_{t|t-1} = \hat{p}_{t-1|t-1}$$

**¿Por qué?** Porque nuestro modelo es un random walk: la mejor predicción de $p_t$ es el valor anterior.

**Predicción de la varianza**:
$$P_{t|t-1} = P_{t-1|t-1} + Q$$

**¿Por qué?** La incertidumbre aumenta debido al ruido del proceso ($Q$).

### **Interpretación Intuitiva**:
- **Predicción**: "Si no tengo nueva información, mi mejor estimación es el valor anterior"
- **Varianza**: "Pero estoy menos seguro porque ha pasado tiempo y el precio puede haber cambiado"

## 2.3 Derivación Paso a Paso: Ecuaciones de Corrección

### **Fase 2: Corrección (Actualización)**

**Dado**: Nueva observación $y_t$ (mid price o trade)

**Ganancia de Kalman**:
$$K_t = \frac{P_{t|t-1}}{P_{t|t-1} + R_t}$$

donde $R_t$ es la varianza del ruido de la observación (depende del tipo).

**Actualización del estado**:
$$\hat{p}_{t|t} = \hat{p}_{t|t-1} + K_t(y_t - \hat{p}_{t|t-1})$$

**Actualización de la varianza**:
$$P_{t|t} = (1 - K_t)P_{t|t-1}$$

### **Interpretación Intuitiva de la Ganancia de Kalman**:

$$K_t = \frac{\text{Incertidumbre de la predicción}}{\text{Incertidumbre total}} = \frac{P_{t|t-1}}{P_{t|t-1} + R_t}$$

- **Si $K_t \approx 1$**: Confío más en la nueva observación (predicción muy incierta)
- **Si $K_t \approx 0$**: Confío más en la predicción (observación muy ruidosa)
- **$K_t$ es automáticamente óptimo**: Minimiza el error cuadrático medio

### **¿Cómo se actualiza la estimación?**
$$\text{Nueva estimación} = \text{Predicción} + \text{Ganancia} \times \text{Innovación}$$

donde **Innovación** = $y_t - \hat{p}_{t|t-1}$ es la "sorpresa" de la observación.

## 2.4 Implementación del Filtro de Kalman - Observaciones

Ahora implementaremos el algoritmo completo paso a paso, empezando con **solo mid prices** como observaciones.

In [10]:
def kalman_filter_basic(observations, Q, R, initial_price, initial_variance):
    """
    Implementación básica del filtro de Kalman para estimación de precios.
    
    Parameters:
    - observations: Array de observaciones (mid_prices)
    - Q: Varianza del proceso (volatilidad del precio verdadero)
    - R: Varianza del ruido de observación
    - initial_price: Estimación inicial del precio
    - initial_variance: Incertidumbre inicial
    
    Returns:
    - estimates: Estimaciones a posteriori del precio verdadero
    - variances: Varianzas a posteriori (incertidumbre)
    - gains: Ganancias de Kalman utilizadas
    """
    n_obs = len(observations)
    
    # Arrays para almacenar resultados
    estimates = np.zeros(n_obs)          # p_{t|t}
    variances = np.zeros(n_obs)          # P_{t|t}
    predictions = np.zeros(n_obs)        # p_{t|t-1}
    pred_variances = np.zeros(n_obs)     # P_{t|t-1}
    gains = np.zeros(n_obs)              # K_t
    innovations = np.zeros(n_obs)        # y_t - p_{t|t-1}
    
    # Inicialización
    estimates[0] = initial_price
    variances[0] = initial_variance
    predictions[0] = initial_price
    pred_variances[0] = initial_variance
    
    # Primera observación
    gains[0] = pred_variances[0] / (pred_variances[0] + R)
    innovations[0] = observations[0] - predictions[0]
    estimates[0] = predictions[0] + gains[0] * innovations[0]
    variances[0] = (1 - gains[0]) * pred_variances[0]
    
    # Iteraciones del filtro
    for t in range(1, n_obs):
        # FASE 1: PREDICCIÓN
        predictions[t] = estimates[t-1]                    # p_{t|t-1} = p_{t-1|t-1}
        pred_variances[t] = variances[t-1] + Q             # P_{t|t-1} = P_{t-1|t-1} + Q
        
        # FASE 2: CORRECCIÓN (si hay observación)
        if not np.isnan(observations[t]):
            # Ganancia de Kalman
            gains[t] = pred_variances[t] / (pred_variances[t] + R)
            
            # Innovación (surpresa de la observación)
            innovations[t] = observations[t] - predictions[t]
            
            # Actualización del estado
            estimates[t] = predictions[t] + gains[t] * innovations[t]
            
            # Actualización de la varianza
            variances[t] = (1 - gains[t]) * pred_variances[t]
        else:
            # Si no hay observación, usar solo la predicción
            estimates[t] = predictions[t]
            variances[t] = pred_variances[t]
            gains[t] = 0
            innovations[t] = 0
    
    return {
        'estimates': estimates,
        'variances': variances,
        'predictions': predictions,
        'pred_variances': pred_variances,
        'gains': gains,
        'innovations': innovations
    }

# Aplicar el filtro de Kalman básico usando solo mid prices
print("🔄 Ejecutando Filtro de Kalman básico (solo mid prices)...")

kalman_results = kalman_filter_basic(
    observations=mid_prices,
    Q=KALMAN_PARAMS['Q'],
    R=KALMAN_PARAMS['R_mid'],
    initial_price=KALMAN_PARAMS['initial_price'],
    initial_variance=KALMAN_PARAMS['initial_variance']
)

print("✅ Filtro de Kalman ejecutado exitosamente")
print(f"📊 Estimaciones generadas para {len(kalman_results['estimates'])} períodos")
print(f"🎯 Ganancia de Kalman promedio: {kalman_results['gains'].mean():.3f}")
print(f"📈 Rango de ganancias: [{kalman_results['gains'].min():.3f}, {kalman_results['gains'].max():.3f}]")
print(f"🔍 Innovación promedio: {kalman_results['innovations'].mean():.4f}")
print(f"📉 Varianza final: {kalman_results['variances'][-1]:.6f}")

🔄 Ejecutando Filtro de Kalman básico (solo mid prices)...
✅ Filtro de Kalman ejecutado exitosamente
📊 Estimaciones generadas para 300 períodos
🎯 Ganancia de Kalman promedio: 0.613
📈 Rango de ganancias: [0.612, 0.906]
🔍 Innovación promedio: 0.0393
📉 Varianza final: 0.023597
✅ Filtro de Kalman ejecutado exitosamente
📊 Estimaciones generadas para 300 períodos
🎯 Ganancia de Kalman promedio: 0.613
📈 Rango de ganancias: [0.612, 0.906]
🔍 Innovación promedio: 0.0393
📉 Varianza final: 0.023597


## 2.5 Extensión: Filtro de Kalman con Observaciones Duales

Ahora extenderemos el filtro para incorporar **trades como segunda fuente de información**. 

### **Innovación Clave**: Varianza Adaptativa según Volumen

Para los trades, la varianza del ruido depende del volumen:

$$R_t^{trade} = \frac{R^{trade}_{base}}{w_t}$$

donde $w_t = \frac{\text{volumen}_t}{\text{volumen promedio}}$ es el peso basado en volumen.

**Intuición**: Trades con mayor volumen contienen más información y tienen menos ruido relativo.

In [11]:
def kalman_filter_dual_observations(mid_prices, trades_data, Q, R_mid, R_trade_base, 
                                   initial_price, initial_variance):
    """
    Filtro de Kalman con observaciones duales: mid prices + trades.
    Incorpora ponderación por volumen para trades.
    
    Parameters:
    - mid_prices: Array de mid prices (siempre disponibles)
    - trades_data: DataFrame con trades (time, price, volume)
    - Q: Varianza del proceso
    - R_mid: Varianza ruido mid prices
    - R_trade_base: Varianza base para trades (se ajusta por volumen)
    - initial_price: Estimación inicial
    - initial_variance: Incertidumbre inicial
    
    Returns:
    - Dictionary con estimaciones y estadísticas del filtro
    """
    n_periods = len(mid_prices)
    
    # Arrays para resultados
    estimates = np.zeros(n_periods)
    variances = np.zeros(n_periods)
    predictions = np.zeros(n_periods)
    pred_variances = np.zeros(n_periods)
    gains_mid = np.zeros(n_periods)
    gains_trade = np.zeros(n_periods)
    innovations_mid = np.zeros(n_periods)
    innovations_trade = np.zeros(n_periods)
    trade_weights = np.zeros(n_periods)
    
    # Crear mapa de trades por tiempo
    trade_map = {}
    avg_volume = trades_data['volume'].mean() if len(trades_data) > 0 else 1000
    
    for _, trade in trades_data.iterrows():
        t = int(trade['time'])
        if t < n_periods:
            trade_map[t] = {
                'price': trade['price'],
                'volume': trade['volume'],
                'weight': trade['volume'] / avg_volume  # Peso basado en volumen
            }
    
    # Inicialización
    estimates[0] = initial_price
    variances[0] = initial_variance
    predictions[0] = initial_price
    pred_variances[0] = initial_variance
    
    # Primera observación (solo mid price)
    gains_mid[0] = pred_variances[0] / (pred_variances[0] + R_mid)
    innovations_mid[0] = mid_prices[0] - predictions[0]
    estimates[0] = predictions[0] + gains_mid[0] * innovations_mid[0]
    variances[0] = (1 - gains_mid[0]) * pred_variances[0]
    
    # Iteraciones principales
    for t in range(1, n_periods):
        # FASE 1: PREDICCIÓN
        predictions[t] = estimates[t-1]
        pred_variances[t] = variances[t-1] + Q
        
        # FASE 2: CORRECCIÓN CON MID PRICE
        # Actualización con mid price (siempre disponible)
        K_mid = pred_variances[t] / (pred_variances[t] + R_mid)
        innovation_mid = mid_prices[t] - predictions[t]
        
        # Estado después de mid price
        estimate_after_mid = predictions[t] + K_mid * innovation_mid
        variance_after_mid = (1 - K_mid) * pred_variances[t]
        
        gains_mid[t] = K_mid
        innovations_mid[t] = innovation_mid
        
        # FASE 3: CORRECCIÓN ADICIONAL CON TRADE (si existe)
        if t in trade_map:
            trade_info = trade_map[t]
            
            # Varianza ajustada por volumen
            R_trade_weighted = R_trade_base / trade_info['weight']
            trade_weights[t] = trade_info['weight']
            
            # Ganancia de Kalman para trade
            K_trade = variance_after_mid / (variance_after_mid + R_trade_weighted)
            innovation_trade = trade_info['price'] - estimate_after_mid
            
            # Estado final después de ambas observaciones
            estimates[t] = estimate_after_mid + K_trade * innovation_trade
            variances[t] = (1 - K_trade) * variance_after_mid
            
            gains_trade[t] = K_trade
            innovations_trade[t] = innovation_trade
        else:
            # Solo mid price
            estimates[t] = estimate_after_mid
            variances[t] = variance_after_mid
            gains_trade[t] = 0
            innovations_trade[t] = 0
    
    return {
        'estimates': estimates,
        'variances': variances,
        'predictions': predictions,
        'pred_variances': pred_variances,
        'gains_mid': gains_mid,
        'gains_trade': gains_trade,
        'innovations_mid': innovations_mid,
        'innovations_trade': innovations_trade,
        'trade_weights': trade_weights,
        'n_trades_used': len([t for t in trade_map.keys() if t < n_periods])
    }

# Aplicar el filtro de Kalman dual
print("🔄 Ejecutando Filtro de Kalman con observaciones duales...")

kalman_dual_results = kalman_filter_dual_observations(
    mid_prices=mid_prices,
    trades_data=trades_data,
    Q=KALMAN_PARAMS['Q'],
    R_mid=KALMAN_PARAMS['R_mid'],
    R_trade_base=KALMAN_PARAMS['R_trade_base'],
    initial_price=KALMAN_PARAMS['initial_price'],
    initial_variance=KALMAN_PARAMS['initial_variance']
)

print("✅ Filtro de Kalman dual ejecutado exitosamente")
print(f"📊 Estimaciones generadas para {len(kalman_dual_results['estimates'])} períodos")
print(f"💱 Trades incorporados: {kalman_dual_results['n_trades_used']}")
print(f"🎯 Ganancia promedio mid prices: {kalman_dual_results['gains_mid'].mean():.3f}")
print(f"📈 Ganancia promedio trades: {kalman_dual_results['gains_trade'][kalman_dual_results['gains_trade']>0].mean():.3f}")
print(f"⚖️ Peso promedio trades: {kalman_dual_results['trade_weights'][kalman_dual_results['trade_weights']>0].mean():.2f}")
print(f"📉 Varianza final: {kalman_dual_results['variances'][-1]:.6f}")

🔄 Ejecutando Filtro de Kalman con observaciones duales...
✅ Filtro de Kalman dual ejecutado exitosamente
📊 Estimaciones generadas para 300 períodos
💱 Trades incorporados: 148
🎯 Ganancia promedio mid prices: 0.550
📈 Ganancia promedio trades: 0.983
⚖️ Peso promedio trades: 1.00
📉 Varianza final: 0.000577


## 2.6 Análisis de Resultados

Comparemos los resultados de ambos filtros de Kalman con el precio verdadero y las observaciones ruidosas.

In [12]:
# Análisis cuantitativo de los resultados
print("📊 ANÁLISIS COMPARATIVO DE RESULTADOS")
print("=" * 60)

# Errores de estimación
time_axis = np.arange(len(true_prices))

# Errores del mid price (línea base)
mid_errors = np.abs(mid_prices - true_prices)
mid_mse = np.mean((mid_prices - true_prices)**2)
mid_mae = np.mean(mid_errors)

# Errores del filtro básico (solo mid prices)
basic_errors = np.abs(kalman_results['estimates'] - true_prices)
basic_mse = np.mean((kalman_results['estimates'] - true_prices)**2)
basic_mae = np.mean(basic_errors)

# Errores del filtro dual (mid prices + trades)
dual_errors = np.abs(kalman_dual_results['estimates'] - true_prices)
dual_mse = np.mean((kalman_dual_results['estimates'] - true_prices)**2)
dual_mae = np.mean(dual_errors)

print(f"📈 PRECISIÓN (Error Absoluto Medio):")
print(f"   • Mid Price (línea base):     ${mid_mae:.4f}")
print(f"   • Kalman Básico:              ${basic_mae:.4f}")
print(f"   • Kalman Dual:                ${dual_mae:.4f}")

print(f"\n📊 ERROR CUADRÁTICO MEDIO:")
print(f"   • Mid Price (línea base):     {mid_mse:.6f}")
print(f"   • Kalman Básico:              {basic_mse:.6f}")
print(f"   • Kalman Dual:                {dual_mse:.6f}")

print(f"\n🎯 MEJORA RELATIVA:")
basic_improvement = (mid_mae - basic_mae) / mid_mae * 100
dual_improvement = (mid_mae - dual_mae) / mid_mae * 100
dual_vs_basic = (basic_mae - dual_mae) / basic_mae * 100

print(f"   • Kalman Básico vs Mid Price: {basic_improvement:.1f}%")
print(f"   • Kalman Dual vs Mid Price:   {dual_improvement:.1f}%")
print(f"   • Kalman Dual vs Básico:      {dual_vs_basic:.1f}%")

# Análisis por períodos de volatilidad
normal_mask = vol_regimes == 0
high_vol_mask = vol_regimes == 1

if high_vol_mask.sum() > 0:
    print(f"\n🔥 ANÁLISIS POR RÉGIMEN DE VOLATILIDAD:")
    
    print(f"   📊 Períodos Normales:")
    print(f"      • Mid Price MAE:   ${mid_errors[normal_mask].mean():.4f}")
    print(f"      • Kalman Dual MAE: ${dual_errors[normal_mask].mean():.4f}")
    
    print(f"   📈 Períodos Alta Volatilidad:")
    print(f"      • Mid Price MAE:   ${mid_errors[high_vol_mask].mean():.4f}")
    print(f"      • Kalman Dual MAE: ${dual_errors[high_vol_mask].mean():.4f}")

# Análisis de incertidumbre
print(f"\n📉 EVOLUCIÓN DE LA INCERTIDUMBRE:")
print(f"   • Varianza inicial:     {KALMAN_PARAMS['initial_variance']:.6f}")
print(f"   • Varianza final básico: {kalman_results['variances'][-1]:.6f}")
print(f"   • Varianza final dual:   {kalman_dual_results['variances'][-1]:.6f}")
print(f"   • Reducción incertidumbre: {(1 - kalman_dual_results['variances'][-1]/KALMAN_PARAMS['initial_variance'])*100:.1f}%")

📊 ANÁLISIS COMPARATIVO DE RESULTADOS
📈 PRECISIÓN (Error Absoluto Medio):
   • Mid Price (línea base):     $0.1561
   • Kalman Básico:              $0.1186
   • Kalman Dual:                $0.0585

📊 ERROR CUADRÁTICO MEDIO:
   • Mid Price (línea base):     0.038579
   • Kalman Básico:              0.022918
   • Kalman Dual:                0.009545

🎯 MEJORA RELATIVA:
   • Kalman Básico vs Mid Price: 24.0%
   • Kalman Dual vs Mid Price:   62.5%
   • Kalman Dual vs Básico:      50.7%

🔥 ANÁLISIS POR RÉGIMEN DE VOLATILIDAD:
   📊 Períodos Normales:
      • Mid Price MAE:   $0.1561
      • Kalman Dual MAE: $0.0666
   📈 Períodos Alta Volatilidad:
      • Mid Price MAE:   $0.1560
      • Kalman Dual MAE: $0.0518

📉 EVOLUCIÓN DE LA INCERTIDUMBRE:
   • Varianza inicial:     0.371675
   • Varianza final básico: 0.023597
   • Varianza final dual:   0.000577
   • Reducción incertidumbre: 99.8%


## 2.7 Visualización Integrada: Comparación Completa de Filtros de Kalman

Esta visualización combina todos los elementos desarrollados para mostrar una comparación comprehensiva del rendimiento de los filtros de Kalman en estimación de precios financieros.

In [13]:
# VISUALIZACIÓN INTEGRADA: Comparación Completa de Filtros de Kalman
# ============================================================================

# Preparar datos para la visualización
time_axis = np.arange(len(true_prices))

# Calcular spread absoluto (ask - bid)
spread_absolute = ask_prices - bid_prices

# Crear array de volúmenes por tiempo (0 si no hay trade)
volumes_by_time = np.zeros(len(true_prices))
for _, trade in trades_data.iterrows():
    t = int(trade['time'])
    if t < len(volumes_by_time):
        volumes_by_time[t] = trade['volume']

# Crear figura con 5 subplots
fig_integrated = make_subplots(
    rows=5, cols=1,
    subplot_titles=(
        '🎯 Precios: Verdadero, Observable (Mid/Bid/Ask) y Estimaciones Kalman',
        '📊 Errores de Estimación vs Spread Bid-Ask',
        '💱 Volúmenes de Trades Ejecutados',
        '⚖️ Ganancia de Kalman en Tiempo Real',
        '🔍 Evolución de la Incertidumbre'
    ),
    vertical_spacing=0.06,
    row_heights=[0.35, 0.2, 0.15, 0.15, 0.15],
    specs=[[{"secondary_y": False}],
           [{"secondary_y": True}],
           [{"secondary_y": False}],
           [{"secondary_y": False}],
           [{"secondary_y": False}]]
)

# =============================================================================
# SUBPLOT 1: PRECIOS (Verdadero + Observable + Filtros Kalman)
# =============================================================================

# Precio verdadero (línea base)
fig_integrated.add_trace(
    go.Scatter(x=time_axis, y=true_prices, 
               name='Precio Verdadero', 
               line=dict(color='white', width=3),
               hovertemplate='<b>Tiempo:</b> %{x}<br><b>Precio Verdadero:</b> $%{y:.4f}<extra></extra>'),
    row=1, col=1
)

# Mid price (observable)
fig_integrated.add_trace(
    go.Scatter(x=time_axis, y=mid_prices, 
               name='Mid Price', 
               line=dict(color='cyan', width=1.5, dash='dot'),
               opacity=0.8,
               hovertemplate='<b>Tiempo:</b> %{x}<br><b>Mid Price:</b> $%{y:.4f}<extra></extra>'),
    row=1, col=1
)

# Área bid-ask (spread visual)
fig_integrated.add_trace(
    go.Scatter(x=time_axis, y=ask_prices, 
               name='Ask Price', 
               line=dict(color='rgba(255,100,100,0.6)', width=1),
               fill=None, mode='lines',
               hovertemplate='<b>Tiempo:</b> %{x}<br><b>Ask:</b> $%{y:.4f}<extra></extra>'),
    row=1, col=1
)

fig_integrated.add_trace(
    go.Scatter(x=time_axis, y=bid_prices, 
               name='Bid Price', 
               line=dict(color='rgba(100,255,100,0.6)', width=1),
               fill='tonexty', fillcolor='rgba(150,150,150,0.15)', mode='lines',
               hovertemplate='<b>Tiempo:</b> %{x}<br><b>Bid:</b> $%{y:.4f}<extra></extra>'),
    row=1, col=1
)

# Filtro de Kalman Básico
fig_integrated.add_trace(
    go.Scatter(x=time_axis, y=kalman_results['estimates'], 
               name='Kalman Básico', 
               line=dict(color='orange', width=2.5),
               hovertemplate='<b>Tiempo:</b> %{x}<br><b>Kalman Básico:</b> $%{y:.4f}<br><b>Error:</b> $%{customdata:.4f}<extra></extra>',
               customdata=basic_errors),
    row=1, col=1
)

# Filtro de Kalman Dual
fig_integrated.add_trace(
    go.Scatter(x=time_axis, y=kalman_dual_results['estimates'], 
               name='Kalman Dual', 
               line=dict(color='lime', width=2.5),
               hovertemplate='<b>Tiempo:</b> %{x}<br><b>Kalman Dual:</b> $%{y:.4f}<br><b>Error:</b> $%{customdata:.4f}<extra></extra>',
               customdata=dual_errors),
    row=1, col=1
)

# Trades como puntos de referencia (muestra reducida para claridad)
trades_sample = trades_data.iloc[::3]  # Cada 3er trade para no saturar
if len(trades_sample) > 0:
    buys_sample = trades_sample[trades_sample['side'] == 1]
    sells_sample = trades_sample[trades_sample['side'] == -1]
    
    if len(buys_sample) > 0:
        fig_integrated.add_trace(
            go.Scatter(x=buys_sample['time'], y=buys_sample['price'], 
                       mode='markers', name='Buy Trades',
                       marker=dict(color='lime', size=5, symbol='triangle-up', line=dict(width=1, color='white')),
                       hovertemplate='<b>Tiempo:</b> %{x}<br><b>Precio Trade:</b> $%{y:.4f}<br><b>Volumen:</b> %{customdata:,.0f}<extra></extra>',
                       customdata=buys_sample['volume']),
            row=1, col=1
        )
    
    if len(sells_sample) > 0:
        fig_integrated.add_trace(
            go.Scatter(x=sells_sample['time'], y=sells_sample['price'], 
                       mode='markers', name='Sell Trades',
                       marker=dict(color='red', size=5, symbol='triangle-down', line=dict(width=1, color='white')),
                       hovertemplate='<b>Tiempo:</b> %{x}<br><b>Precio Trade:</b> $%{y:.4f}<br><b>Volumen:</b> %{customdata:,.0f}<extra></extra>',
                       customdata=sells_sample['volume']),
            row=1, col=1
        )

print("✅ Subplot 1 (Precios) configurado")

✅ Subplot 1 (Precios) configurado


In [14]:
# =============================================================================
# SUBPLOT 2: ERRORES vs SPREAD (Eje dual)
# =============================================================================

# Errores de estimación (eje izquierdo)
fig_integrated.add_trace(
    go.Scatter(x=time_axis, y=basic_errors, 
               name='Error Kalman Básico', 
               line=dict(color='orange', width=2),
               yaxis='y2',
               hovertemplate='<b>Tiempo:</b> %{x}<br><b>Error Básico:</b> $%{y:.4f}<extra></extra>'),
    row=2, col=1
)

fig_integrated.add_trace(
    go.Scatter(x=time_axis, y=dual_errors, 
               name='Error Kalman Dual', 
               line=dict(color='lime', width=2),
               yaxis='y2',
               hovertemplate='<b>Tiempo:</b> %{x}<br><b>Error Dual:</b> $%{y:.4f}<extra></extra>'),
    row=2, col=1
)

# Spread bid-ask (eje derecho)
fig_integrated.add_trace(
    go.Scatter(x=time_axis, y=spread_absolute, 
               name='Spread (Ask-Bid)', 
               line=dict(color='yellow', width=1.5, dash='dash'),
               yaxis='y3',
               opacity=0.7,
               hovertemplate='<b>Tiempo:</b> %{x}<br><b>Spread:</b> $%{y:.4f}<extra></extra>'),
    row=2, col=1
)

# =============================================================================
# SUBPLOT 3: VOLÚMENES DE TRADES
# =============================================================================

# Barras horizontales para volúmenes (cuando hay trades)
trade_colors = []
trade_texts = []
for i, vol in enumerate(volumes_by_time):
    if vol > 0:
        # Buscar el lado del trade en ese momento
        trade_at_time = trades_data[trades_data['time'] == i]
        if len(trade_at_time) > 0:
            side = trade_at_time.iloc[0]['side']
            if side == 1:
                trade_colors.append('lime')
                trade_texts.append(f'Buy: {vol:,.0f}')
            else:
                trade_colors.append('red')
                trade_texts.append(f'Sell: {vol:,.0f}')
        else:
            trade_colors.append('gray')
            trade_texts.append(f'Trade: {vol:,.0f}')
    else:
        trade_colors.append('rgba(100,100,100,0.1)')
        trade_texts.append('No trade')

fig_integrated.add_trace(
    go.Bar(x=time_axis, y=volumes_by_time,
           name='Volumen Trades',
           marker=dict(color=trade_colors, line=dict(width=0.5, color='white')),
           text=trade_texts,
           textposition='outside',
           textfont=dict(size=8),
           hovertemplate='<b>Tiempo:</b> %{x}<br><b>Volumen:</b> %{y:,.0f}<br>%{text}<extra></extra>',
           showlegend=False),
    row=3, col=1
)

# =============================================================================
# SUBPLOT 4: GANANCIA DE KALMAN
# =============================================================================

# Ganancia Kalman para mid prices (siempre activa)
fig_integrated.add_trace(
    go.Scatter(x=time_axis, y=kalman_dual_results['gains_mid'], 
               name='Ganancia Mid Price', 
               line=dict(color='cyan', width=2),
               fill='tozeroy', fillcolor='rgba(0,255,255,0.2)',
               hovertemplate='<b>Tiempo:</b> %{x}<br><b>Ganancia Mid:</b> %{y:.3f}<extra></extra>'),
    row=4, col=1
)

# Ganancia Kalman para trades (solo cuando hay trades)
trades_gain_x = []
trades_gain_y = []
for i, gain in enumerate(kalman_dual_results['gains_trade']):
    if gain > 0:  # Solo cuando hay trade
        trades_gain_x.append(i)
        trades_gain_y.append(gain)

if len(trades_gain_x) > 0:
    fig_integrated.add_trace(
        go.Scatter(x=trades_gain_x, y=trades_gain_y, 
                   mode='markers+lines',
                   name='Ganancia Trades', 
                   line=dict(color='orange', width=1),
                   marker=dict(color='orange', size=6),
                   hovertemplate='<b>Tiempo:</b> %{x}<br><b>Ganancia Trade:</b> %{y:.3f}<extra></extra>'),
        row=4, col=1
    )

# =============================================================================
# SUBPLOT 5: EVOLUCIÓN DE LA INCERTIDUMBRE
# =============================================================================

# Incertidumbre del filtro básico
uncertainty_basic = np.sqrt(kalman_results['variances'])
fig_integrated.add_trace(
    go.Scatter(x=time_axis, y=uncertainty_basic, 
               name='Incertidumbre Básico', 
               line=dict(color='orange', width=2),
               fill='tozeroy', fillcolor='rgba(255,165,0,0.2)',
               hovertemplate='<b>Tiempo:</b> %{x}<br><b>Incertidumbre Básico:</b> %{y:.4f}<extra></extra>'),
    row=5, col=1
)

# Incertidumbre del filtro dual
uncertainty_dual = np.sqrt(kalman_dual_results['variances'])
fig_integrated.add_trace(
    go.Scatter(x=time_axis, y=uncertainty_dual, 
               name='Incertidumbre Dual', 
               line=dict(color='lime', width=2),
               fill='tozeroy', fillcolor='rgba(0,255,0,0.15)',
               hovertemplate='<b>Tiempo:</b> %{x}<br><b>Incertidumbre Dual:</b> %{y:.4f}<extra></extra>'),
    row=5, col=1
)

print("✅ Subplots 2-5 configurados")

✅ Subplots 2-5 configurados


In [15]:
# =============================================================================
# CONFIGURACIÓN FINAL DEL LAYOUT Y EJES
# =============================================================================

# Layout principal con tema oscuro
fig_integrated.update_layout(
    title={
        'text': '🎯 Análisis Integrado: Filtros de Kalman para Estimación de Precios Financieros',
        'font': {'color': 'white', 'size': 22, 'family': 'Arial Black'},
        'x': 0.5
    },
    height=1200,
    showlegend=True,
    legend=dict(
        x=1.02, y=1,
        font=dict(color='white', size=10),
        bgcolor='rgba(17, 17, 17, 0.9)',
        bordercolor='rgba(255, 255, 255, 0.3)',
        borderwidth=1
    ),
    plot_bgcolor='rgb(17, 17, 17)',
    paper_bgcolor='rgb(17, 17, 17)',
    font=dict(color='white', family='Arial')
)

# Configurar ejes con tema oscuro para todos los subplots
# CORRECCIÓN: Alineación perfecta de todos los ejes X
for i in range(1, 6):
    fig_integrated.update_xaxes(
        gridcolor='rgb(68, 68, 68)',
        tickfont=dict(color='white', size=10),
        title_font=dict(color='white', size=11),
        range=[-0.5, len(time_axis)-0.5],  # Rango explícito para alineación perfecta
        dtick=50,  # Marcas cada 50 períodos para alineación visual
        row=i, col=1
    )
    fig_integrated.update_yaxes(
        gridcolor='rgb(68, 68, 68)',
        tickfont=dict(color='white', size=10),
        title_font=dict(color='white', size=11),
        row=i, col=1
    )

# Títulos específicos de ejes
fig_integrated.update_xaxes(title_text="Tiempo (minutos)", row=5, col=1)

fig_integrated.update_yaxes(title_text="Precio ($)", row=1, col=1)
fig_integrated.update_yaxes(title_text="Error ($)", row=2, col=1)
fig_integrated.update_yaxes(title_text="Volumen", row=3, col=1)
fig_integrated.update_yaxes(title_text="Ganancia K", row=4, col=1)
fig_integrated.update_yaxes(title_text="Incertidumbre σ", row=5, col=1)

# Configurar eje secundario para el spread (subplot 2)
fig_integrated.update_yaxes(
    title_text="Spread ($)", 
    side="right",
    overlaying="y2",
    row=2, col=1,
    secondary_y=True
)

# Actualizar títulos de subplots con color blanco
annotations = list(fig_integrated.layout.annotations)
for annotation in annotations:
    annotation.font.color = 'white'
    annotation.font.size = 12
fig_integrated.layout.annotations = annotations

# Ajustar rangos de algunos ejes para mejor visualización
fig_integrated.update_yaxes(range=[0, max(volumes_by_time)*1.1], row=3, col=1)  # Volúmenes
fig_integrated.update_yaxes(range=[0, 1], row=4, col=1)  # Ganancia (0-1)

# Mostrar la visualización
fig_integrated.show()

print("🎯 VISUALIZACIÓN INTEGRADA COMPLETADA")
print("=" * 60)
print("📊 La visualización incluye:")
print("   1️⃣ Precios: Verdadero, observables (Mid/Bid/Ask), trades y ambos filtros Kalman")
print("   2️⃣ Errores: Comparación de errores de ambos filtros vs spread bid-ask")
print("   3️⃣ Volúmenes: Barras de volumen cuando hay trades (0 cuando no hay)")
print("   4️⃣ Ganancias: Evolución de las ganancias de Kalman en tiempo real")
print("   5️⃣ Incertidumbre: Reducción de varianza de ambos filtros en el tiempo")
print()
print("🔍 OBSERVACIONES CLAVE:")
print(f"   • Kalman Dual (verde) vs Básico (naranja) en seguimiento del precio verdadero")
print(f"   • Correlación entre spread alto y errores de estimación")
print(f"   • Volúmenes de trades distribuidos temporalmente")
print(f"   • Ganancias altas inicial que se estabilizan (~0.94 para mid prices)")
print(f"   • Reducción dramática de incertidumbre en primeros períodos")
print()
print("💡 Interactividad: Hover sobre cualquier punto para ver detalles específicos")

🎯 VISUALIZACIÓN INTEGRADA COMPLETADA
📊 La visualización incluye:
   1️⃣ Precios: Verdadero, observables (Mid/Bid/Ask), trades y ambos filtros Kalman
   2️⃣ Errores: Comparación de errores de ambos filtros vs spread bid-ask
   3️⃣ Volúmenes: Barras de volumen cuando hay trades (0 cuando no hay)
   4️⃣ Ganancias: Evolución de las ganancias de Kalman en tiempo real
   5️⃣ Incertidumbre: Reducción de varianza de ambos filtros en el tiempo

🔍 OBSERVACIONES CLAVE:
   • Kalman Dual (verde) vs Básico (naranja) en seguimiento del precio verdadero
   • Correlación entre spread alto y errores de estimación
   • Volúmenes de trades distribuidos temporalmente
   • Ganancias altas inicial que se estabilizan (~0.94 para mid prices)
   • Reducción dramática de incertidumbre en primeros períodos

💡 Interactividad: Hover sobre cualquier punto para ver detalles específicos


3. Kalman Filter Multi Asset 

# 3. Filtro de Kalman Multiasset 🔗

En mercados financieros reales, los activos no se mueven de forma independiente. Existe **correlación** entre precios de activos similares (acciones del mismo sector, bonos de diferentes vencimientos, etc.). Además, la información puede ser **escasa** - no siempre hay trades disponibles para todos los activos.

El **Filtro de Kalman Multiasset** aprovecha estas correlaciones para mejorar las estimaciones cuando la información directa es limitada. Si el Activo A no tiene trades recientes, pero el Activo B (correlacionado) sí los tiene, podemos usar esa información para mejorar nuestra estimación del Activo A.

## 3.1 Marco Teórico: Vector de Estados

### **Modelo Matemático Extendido**

**Vector de Estados** (2 activos):
$$\mathbf{x}_t = \begin{bmatrix} p_t^A \\ p_t^B \end{bmatrix}$$

**Ecuación de Transición** (con correlación):
$$\mathbf{x}_{t+1} = \mathbf{F} \mathbf{x}_t + \mathbf{w}_t$$

donde:
$$\mathbf{F} = \begin{bmatrix} 1 & \beta_{A \leftarrow B} \\ \beta_{B \leftarrow A} & 1 \end{bmatrix}, \quad \mathbf{w}_t \sim N(\mathbf{0}, \mathbf{Q})$$

$$\mathbf{Q} = \begin{bmatrix} Q_{AA} & Q_{AB} \\ Q_{AB} & Q_{BB} \end{bmatrix}$$

### **Interpretación:**
- $\beta_{A \leftarrow B}$: Influencia del Activo B sobre A
- $Q_{AB}$: Covarianza entre innovaciones de precios
- **Observaciones cruzadas**: Trade del Activo A informa sobre el Activo B

### **Ventajas del Filtro Multiasset:**
1. **Información compartida**: Los trades de un activo informan sobre el otro
2. **Mejor estimación**: Durante períodos sin trades, el filtro usa información del activo correlacionado
3. **Reducción de incertidumbre**: La matriz de covarianza se aprovecha para mejorar ambas estimaciones
4. **Robustez**: Menos dependiente de la disponibilidad de trades individuales

---

## 3.2 Generación de Activos Correlacionados

Para simular un entorno realista, generaremos dos activos con correlación controlada. El **Activo B** tendrá movimientos correlacionados con el **Activo A**, pero con su propia volatilidad y régimen específico.

In [16]:
def generate_correlated_asset(true_prices_A, vol_regimes_A, correlation=0.99, vol_ratio=0.9, initial_price_B=102.0):
    """
    Genera un segundo activo con correlación casi perfecta (99%) con el primero.
    CORREGIDO: Método Cholesky para garantizar correlación POSITIVA exacta.
    
    Parameters:
    - true_prices_A: Precios verdaderos del activo A
    - vol_regimes_A: Régimen de volatilidad del activo A
    - correlation: Correlación entre ambos activos (0.99 = 99% POSITIVA)
    - vol_ratio: Ratio de volatilidad B/A (0.9 = ligeramente menos volátil)
    - initial_price_B: Precio inicial del activo B
    
    Returns:
    - true_prices_B: Precios verdaderos del activo B
    - vol_regimes_B: Régimen de volatilidad del activo B (idéntico al A)
    """
    n_periods = len(true_prices_A)
    
    # B tendrá EXACTAMENTE el mismo régimen de volatilidad que A
    vol_regimes_B = vol_regimes_A.copy()
    
    # Calcular rendimientos del activo A
    returns_A = np.diff(np.log(true_prices_A))
    
    # MÉTODO CHOLESKY CORREGIDO para correlación casi perfecta
    np.random.seed(142)  # Seed específico para reproducibilidad
    independent_noise = np.random.normal(0, 1, n_periods - 1)
    
    # Crear returns perfectamente correlacionados
    returns_B = np.zeros(n_periods - 1)
    for t in range(n_periods - 1):
        # B = 99% * A + 1% * independiente (método directo)
        returns_B[t] = correlation * returns_A[t] + np.sqrt(1 - correlation**2) * independent_noise[t] * np.std(returns_A) * vol_ratio
    
    # Construir serie de precios del activo B
    log_prices_B = np.zeros(n_periods)
    log_prices_B[0] = np.log(initial_price_B)
    
    for t in range(1, n_periods):
        log_prices_B[t] = log_prices_B[t-1] + returns_B[t-1]
    
    true_prices_B = np.exp(log_prices_B)
    
    return true_prices_B, vol_regimes_B

# Parámetros optimizados para demostrar superioridad multiasset
MULTIASSET_PARAMS = {
    'correlation': 0.99,          # 99% correlación POSITIVA casi perfecta
    'vol_ratio': 0.9,             # B ligeramente menos volátil que A
    'initial_price_B': 102.0,     # Precio inicial similar
    'trade_frequency_A': 1/40,    # Activo A: muy pocos trades (alta incertidumbre)
    'trade_frequency_B': 25/40,   # Activo B: muchos trades (baja incertidumbre)
    'spread_multiplier_A': 10.0,  # Activo A: spreads ENORMES (10x más grandes)
    'spread_multiplier_B': 8.0,   # Activo B: spreads también grandes (8x)
    'mid_noise_A': 0.01,          # Activo A: mid price muy ruidoso (1%)
    'mid_noise_B': 0.008,         # Activo B: mid price ruidoso (0.8%)
    'trade_noise_A': 0.0001,      # Activo A: trades precisos cuando ocurren
    'trade_noise_B': 0.0001       # Activo B: trades precisos
}

# Generar activo B con correlación casi perfecta
print("🔄 Generando activo B con correlación 99% POSITIVA...")

true_prices_B, vol_regimes_B = generate_correlated_asset(
    true_prices, vol_regimes, 
    correlation=MULTIASSET_PARAMS['correlation'],
    vol_ratio=MULTIASSET_PARAMS['vol_ratio'],
    initial_price_B=MULTIASSET_PARAMS['initial_price_B']
)

print("🔗 ACTIVO B GENERADO CON CORRELACIÓN CASI PERFECTA")
print(f"📈 Precio inicial: ${MULTIASSET_PARAMS['initial_price_B']:.2f}")
print(f"📉 Precio final: ${true_prices_B[-1]:.2f}")
print(f"📊 Rendimiento total: {((true_prices_B[-1]/MULTIASSET_PARAMS['initial_price_B'])-1)*100:.1f}%")
print(f"🔗 Correlación configurada: {MULTIASSET_PARAMS['correlation']*100:.0f}%")
print(f"📈 Volatilidad relativa: {MULTIASSET_PARAMS['vol_ratio']*100:.0f}%")

# Verificar correlación real
returns_A_real = np.diff(np.log(true_prices))
returns_B_real = np.diff(np.log(true_prices_B))
actual_correlation = np.corrcoef(returns_A_real, returns_B_real)[0, 1]
print(f"✅ Correlación real obtenida: {actual_correlation:.4f}")

# Estadísticas de régimen de volatilidad
high_vol_periods_B = np.sum(vol_regimes_B == 1)
print(f"🔥 Períodos alta volatilidad B: {high_vol_periods_B} de {len(vol_regimes_B)} ({high_vol_periods_B/len(vol_regimes_B)*100:.1f}%)")
print(f"🎯 Régimen B idéntico a A: {np.array_equal(vol_regimes, vol_regimes_B)}")

🔄 Generando activo B con correlación 99% POSITIVA...
🔗 ACTIVO B GENERADO CON CORRELACIÓN CASI PERFECTA
📈 Precio inicial: $102.00
📉 Precio final: $109.53
📊 Rendimiento total: 7.4%
🔗 Correlación configurada: 99%
📈 Volatilidad relativa: 90%
✅ Correlación real obtenida: 0.9905
🔥 Períodos alta volatilidad B: 164 de 300 (54.7%)
🎯 Régimen B idéntico a A: True


## 3.3 Microestructura con Trades Escasos

Para simular un entorno donde la información es limitada, reduciremos drásticamente la frecuencia de trades a **1/20** de la frecuencia original. Esto creará períodos largos sin información directa, donde el filtro multiasset debe aprovechar la correlación entre activos.

In [17]:
# Funciones especializadas para microestructura diferenciada

def generate_bid_ask_data_custom(true_prices, vol_regimes, spread_multiplier=1.0, mid_noise_pct=0.002):
    """
    Genera bid/ask/mid con parámetros personalizados para cada activo.
    """
    n_ticks = len(true_prices)
    base_spread_pct = SIMULATION_PARAMS['base_spread_pct'] * spread_multiplier
    max_spread_pct = SIMULATION_PARAMS['max_spread_pct'] * spread_multiplier
    
    spreads_pct = np.zeros(n_ticks)
    bid_prices = np.zeros(n_ticks)
    ask_prices = np.zeros(n_ticks)
    mid_prices = np.zeros(n_ticks)
    volumes_bid = np.zeros(n_ticks)
    volumes_ask = np.zeros(n_ticks)
    
    for t in range(n_ticks):
        # Spread dinámico basado en volatilidad
        if vol_regimes[t] == 1:  # Alta volatilidad
            spread_pct = base_spread_pct + (max_spread_pct - base_spread_pct) * np.random.uniform(0.5, 1.0)
        else:  # Volatilidad normal
            spread_pct = base_spread_pct * np.random.uniform(0.8, 1.2)
        
        spreads_pct[t] = spread_pct
        
        # Mid price con ruido específico respecto al precio verdadero
        mid_price_noise = np.random.normal(0, true_prices[t] * mid_noise_pct)
        mid_prices[t] = true_prices[t] + mid_price_noise
        
        # Bid/ask centrados en mid price (no precio verdadero)
        half_spread = mid_prices[t] * spread_pct / 2
        bid_prices[t] = mid_prices[t] - half_spread
        ask_prices[t] = mid_prices[t] + half_spread
        
        # Volúmenes
        base_volume = SIMULATION_PARAMS['base_volume']
        if vol_regimes[t] == 1:
            volume_mean = base_volume * 0.7  # Menor liquidez durante alta volatilidad
        else:
            volume_mean = base_volume
            
        volumes_bid[t] = np.random.lognormal(np.log(volume_mean) - 0.5 * 0.5**2, 0.5)
        volumes_ask[t] = np.random.lognormal(np.log(volume_mean) - 0.5 * 0.5**2, 0.5)
    
    return bid_prices, ask_prices, spreads_pct, mid_prices, volumes_bid, volumes_ask

def generate_trades_custom(mid_prices, vol_regimes, true_prices, trade_frequency, trade_noise_pct=0.0001):
    """
    Genera trades con frecuencia y ruido personalizados.
    """
    n_ticks = len(mid_prices)
    base_volume = SIMULATION_PARAMS['base_volume']
    volume_std = SIMULATION_PARAMS['volume_std']
    
    trade_times = []
    trade_prices = []
    trade_volumes = []
    trade_sides = []
    
    for t in range(n_ticks):
        # Frecuencia de trades ajustada
        if vol_regimes[t] == 1:  # Alta volatilidad
            freq = trade_frequency * 2.0
        else:  # Volatilidad normal
            freq = trade_frequency
        
        # ¿Ocurre un trade en este tick?
        if np.random.random() < freq:
            trade_times.append(t)
            
            # Lado del trade (aleatorio)
            side = np.random.choice([1, -1])  # 1=buy, -1=sell
            trade_sides.append(side)
            
            # Volumen del trade
            if vol_regimes[t] == 1:
                volume_mean = base_volume * 1.5  # Mayor volumen durante volatilidad
            else:
                volume_mean = base_volume
                
            volume = np.random.lognormal(
                np.log(volume_mean) - 0.5 * (volume_std/volume_mean)**2,
                volume_std / volume_mean
            )
            trade_volumes.append(int(volume))
            
            # PRECIO BASADO EN PRECIO VERDADERO (muy cerca)
            volume_weight = min(volume / (base_volume * 2), 3.0)
            adjusted_noise_pct = trade_noise_pct / np.sqrt(volume_weight)
            
            price_noise = np.random.normal(0, true_prices[t] * adjusted_noise_pct)
            side_bias = side * true_prices[t] * 0.00005  # Bias mínimo por lado
            
            trade_price = true_prices[t] + price_noise + side_bias
            trade_prices.append(trade_price)
    
    return np.array(trade_times), np.array(trade_prices), np.array(trade_volumes), np.array(trade_sides)

print("🔄 Generando microestructura DIFERENCIADA...")

# =============================================================================
# ACTIVO A: Spread grande, mid price ruidoso, pocos trades
# =============================================================================
print("📈 ACTIVO A: Spread grande, mid price ruidoso, MUY pocos trades")

bid_prices_A, ask_prices_A, spreads_pct_A, mid_prices_A, volumes_bid_A, volumes_ask_A = generate_bid_ask_data_custom(
    true_prices, vol_regimes, 
    spread_multiplier=MULTIASSET_PARAMS['spread_multiplier_A'],
    mid_noise_pct=MULTIASSET_PARAMS['mid_noise_A']
)

trade_times_A, trade_prices_A, trade_volumes_A, trade_sides_A = generate_trades_custom(
    mid_prices_A, vol_regimes, true_prices,
    trade_frequency=MULTIASSET_PARAMS['trade_frequency_A'],
    trade_noise_pct=MULTIASSET_PARAMS['trade_noise_A']
)

print(f"✅ Activo A: {len(trade_times_A)} trades (frecuencia: {MULTIASSET_PARAMS['trade_frequency_A']:.4f})")
print(f"   📊 Spread promedio: {spreads_pct_A.mean()*10000:.1f} bps")
print(f"   📈 Ruido mid price: {MULTIASSET_PARAMS['mid_noise_A']*100:.2f}%")

# =============================================================================  
# ACTIVO B: Spreads GRANDES también, mid price ruidoso, MUCHOS trades
# =============================================================================
print("📉 ACTIVO B: Spreads GRANDES también, mid price ruidoso, MUCHOS trades")

bid_prices_B, ask_prices_B, spreads_pct_B, mid_prices_B, volumes_bid_B, volumes_ask_B = generate_bid_ask_data_custom(
    true_prices_B, vol_regimes_B, 
    spread_multiplier=MULTIASSET_PARAMS['spread_multiplier_B'],  # Spreads grandes también
    mid_noise_pct=MULTIASSET_PARAMS['mid_noise_B']
)

trade_times_B, trade_prices_B, trade_volumes_B, trade_sides_B = generate_trades_custom(
    mid_prices_B, vol_regimes_B, true_prices_B,
    trade_frequency=MULTIASSET_PARAMS['trade_frequency_B'],
    trade_noise_pct=MULTIASSET_PARAMS['trade_noise_B']
)

print(f"✅ Activo B: {len(trade_times_B)} trades (frecuencia: {MULTIASSET_PARAMS['trade_frequency_B']:.4f})")
print(f"   📊 Spread promedio: {spreads_pct_B.mean()*10000:.1f} bps")
print(f"   📈 Ruido mid price: {MULTIASSET_PARAMS['mid_noise_B']*100:.2f}%")

# Crear DataFrames
market_data_A = pd.DataFrame({
    'time': range(len(true_prices)),
    'true_price': true_prices,
    'mid_price': mid_prices_A,
    'bid_price': bid_prices_A,
    'ask_price': ask_prices_A,
    'spread_pct': spreads_pct_A,
    'spread_bps': spreads_pct_A * 10000,
    'vol_regime': vol_regimes,
    'volume_bid': volumes_bid_A,
    'volume_ask': volumes_ask_A
})

market_data_B = pd.DataFrame({
    'time': range(len(true_prices_B)),
    'true_price': true_prices_B,
    'mid_price': mid_prices_B,
    'bid_price': bid_prices_B,
    'ask_price': ask_prices_B,
    'spread_pct': spreads_pct_B,
    'spread_bps': spreads_pct_B * 10000,
    'vol_regime': vol_regimes_B,
    'volume_bid': volumes_bid_B,
    'volume_ask': volumes_ask_B
})

trades_data_A = pd.DataFrame({
    'time': trade_times_A,
    'price': trade_prices_A,
    'volume': trade_volumes_A,
    'side': trade_sides_A,
    'side_str': ['buy' if s == 1 else 'sell' for s in trade_sides_A]
})

trades_data_B = pd.DataFrame({
    'time': trade_times_B,
    'price': trade_prices_B,
    'volume': trade_volumes_B,
    'side': trade_sides_B,
    'side_str': ['buy' if s == 1 else 'sell' for s in trade_sides_B]
})

print("\\n📊 RESUMEN MULTIASSET DIFERENCIADO:")
print(f"📈 Activo A - Trades: {len(trades_data_A)} (cobertura: {len(trades_data_A)/300*100:.1f}%)")
print(f"📉 Activo B - Trades: {len(trades_data_B)} (cobertura: {len(trades_data_B)/300*100:.1f}%)")
print(f"🔗 Ratio trades B/A: {len(trades_data_B)/max(len(trades_data_A),1):.1f}x")

# Calcular períodos con al menos un trade (cualquier activo)
periods_with_any_trade = len(set(trade_times_A) | set(trade_times_B))
print(f"🔗 Períodos con al menos un trade (A o B): {periods_with_any_trade}/300 ({periods_with_any_trade/300*100:.1f}%)")

# Estadística de correlación entre precios verdaderos
correlation_true_prices = np.corrcoef(true_prices, true_prices_B)[0, 1]
print(f"🎯 Correlación precios verdaderos: {correlation_true_prices:.3f}")

# Análisis de ruido en observaciones
mid_noise_A = np.abs(mid_prices_A - true_prices)
mid_noise_B = np.abs(mid_prices_B - true_prices_B)
print(f"\\n🔍 ANÁLISIS DE RUIDO:")
print(f"📈 MAE Mid Price A vs True: ${np.mean(mid_noise_A):.4f}")
print(f"📉 MAE Mid Price B vs True: ${np.mean(mid_noise_B):.4f}")

if len(trades_data_A) > 0:
    trade_noise_A = np.abs(trade_prices_A - true_prices[trade_times_A])
    print(f"📈 MAE Trades A vs True: ${np.mean(trade_noise_A):.4f}")
    
if len(trades_data_B) > 0:
    trade_noise_B = np.abs(trade_prices_B - true_prices_B[trade_times_B])
    print(f"📉 MAE Trades B vs True: ${np.mean(trade_noise_B):.4f}")

print("✅ Microestructura multiasset diferenciada generada")

🔄 Generando microestructura DIFERENCIADA...
📈 ACTIVO A: Spread grande, mid price ruidoso, MUY pocos trades
✅ Activo A: 12 trades (frecuencia: 0.0250)
   📊 Spread promedio: 1119.7 bps
   📈 Ruido mid price: 1.00%
📉 ACTIVO B: Spreads GRANDES también, mid price ruidoso, MUCHOS trades
✅ Activo B: 248 trades (frecuencia: 0.6250)
   📊 Spread promedio: 887.6 bps
   📈 Ruido mid price: 0.80%
\n📊 RESUMEN MULTIASSET DIFERENCIADO:
📈 Activo A - Trades: 12 (cobertura: 4.0%)
📉 Activo B - Trades: 248 (cobertura: 82.7%)
🔗 Ratio trades B/A: 20.7x
🔗 Períodos con al menos un trade (A o B): 251/300 (83.7%)
🎯 Correlación precios verdaderos: 1.000
\n🔍 ANÁLISIS DE RUIDO:
📈 MAE Mid Price A vs True: $0.8167
📉 MAE Mid Price B vs True: $0.7149
📈 MAE Trades A vs True: $0.0100
📉 MAE Trades B vs True: $0.0125
✅ Microestructura multiasset diferenciada generada


## 3.4 Implementación del Filtro de Kalman Multiasset

### **Marco Matemático Completo**

**Vector de Estados (2D)**:
$$\mathbf{x}_t = \begin{bmatrix} p_t^A \\ p_t^B \end{bmatrix}$$

**Ecuación de Transición**:
$$\mathbf{x}_{t+1} = \mathbf{F} \mathbf{x}_t + \mathbf{w}_t$$

donde:
$$\mathbf{F} = \begin{bmatrix} 1 & 0 \\ 0 & 1 \end{bmatrix}, \quad \mathbf{w}_t \sim N(\mathbf{0}, \mathbf{Q})$$

**Matriz de Covarianza del Proceso**:
$$\mathbf{Q} = \begin{bmatrix} Q_{AA} & Q_{AB} \\ Q_{AB} & Q_{BB} \end{bmatrix}$$

**Observaciones Disponibles**:
1. **Mid Price A**: $y_t^{mid,A} = p_t^A + v_t^{mid,A}$
2. **Mid Price B**: $y_t^{mid,B} = p_t^B + v_t^{mid,B}$  
3. **Trade A**: $y_t^{trade,A} = p_t^A + v_t^{trade,A}$ (cuando disponible)
4. **Trade B**: $y_t^{trade,B} = p_t^B + v_t^{trade,B}$ (cuando disponible)
5. **Cross-observations**: Trade A también informa sobre precio B via correlación

### **Ventaja Clave**:
Cuando el Activo A no tiene trades, pero el Activo B sí, el filtro puede **inferir información sobre A** usando:
- La correlación conocida entre activos
- El trade observado en B
- La matriz de covarianza cruzada

In [18]:
def kalman_filter_multiasset(market_data_A, trades_data_A, market_data_B, trades_data_B, params):
    """
    Filtro de Kalman para múltiples activos correlacionados.
    
    Parameters:
    - market_data_A, market_data_B: DataFrames con mid prices de ambos activos
    - trades_data_A, trades_data_B: DataFrames con trades de ambos activos  
    - params: Diccionario con parámetros del filtro
    
    Returns:
    - Diccionario con estimaciones y métricas para ambos activos
    """
    n_periods = len(market_data_A)
    
    # Inicialización del vector de estados 2D
    state_estimates = np.zeros((n_periods, 2))  # [precio_A, precio_B]
    state_variances = np.zeros((n_periods, 2, 2))  # Matriz covarianza 2x2
    
    # Arrays para tracking
    innovations_mid_A = np.zeros(n_periods)
    innovations_mid_B = np.zeros(n_periods)
    innovations_trade_A = np.zeros(n_periods)
    innovations_trade_B = np.zeros(n_periods)
    gains_mid_A = np.zeros(n_periods)
    gains_mid_B = np.zeros(n_periods)
    gains_trade_A = np.zeros(n_periods)
    gains_trade_B = np.zeros(n_periods)
    
    # Matrices del sistema
    F = np.eye(2)  # Matriz de transición (identidad para random walk)
    
    # Matriz Q: covarianza del proceso con correlación POSITIVA CORREGIDA
    correlation_AB = 0.99  # Usamos correlación POSITIVA alta
    Q_AA = params['Q_A'] 
    Q_BB = params['Q_B']
    Q_AB = correlation_AB * np.sqrt(Q_AA * Q_BB)  # Correlación POSITIVA
    
    Q = np.array([[Q_AA, Q_AB], 
                  [Q_AB, Q_BB]])
    
    # Crear mapas de trades por tiempo
    trade_map_A = {}
    trade_map_B = {}
    
    for _, trade in trades_data_A.iterrows():
        t = int(trade['time'])
        if t < n_periods:
            trade_map_A[t] = {
                'price': trade['price'],
                'volume': trade['volume'],
                'R': params['R_trade_base'] / np.sqrt(trade['volume'] / 1000)
            }
    
    for _, trade in trades_data_B.iterrows():
        t = int(trade['time'])
        if t < n_periods:
            trade_map_B[t] = {
                'price': trade['price'],
                'volume': trade['volume'],
                'R': params['R_trade_base'] / np.sqrt(trade['volume'] / 1000)
            }
    
    # Estado inicial
    state_estimates[0] = [params['initial_price_A'], params['initial_price_B']]
    state_variances[0] = np.eye(2) * params['initial_variance']
    
    # Loop principal del filtro
    for t in range(1, n_periods):
        # ============ PREDICCIÓN ============
        # Estado predicho
        state_pred = F @ state_estimates[t-1]
        # Covarianza predicha  
        P_pred = F @ state_variances[t-1] @ F.T + Q
        
        # ============ CORRECCIÓN ============
        state_corr = state_pred.copy()
        P_corr = P_pred.copy()
        
        # Lista para acumular observaciones múltiples
        observations = []
        H_matrices = []
        R_matrices = []
        
        # Mid price del activo A (siempre disponible)
        observations.append(market_data_A.loc[t, 'mid_price'])
        H_matrices.append([1, 0])  # Observa solo precio A
        R_matrices.append(params['R_mid_A'])
        
        # Mid price del activo B (siempre disponible)
        observations.append(market_data_B.loc[t, 'mid_price'])
        H_matrices.append([0, 1])  # Observa solo precio B
        R_matrices.append(params['R_mid_B'])
        
        # Trade del activo A (si disponible)
        if t in trade_map_A:
            observations.append(trade_map_A[t]['price'])
            H_matrices.append([1, 0])  # Observa precio A
            R_matrices.append(trade_map_A[t]['R'])
        
        # Trade del activo B (si disponible)
        if t in trade_map_B:
            observations.append(trade_map_B[t]['price'])
            H_matrices.append([0, 1])  # Observa precio B
            R_matrices.append(trade_map_B[t]['R'])
        
        # Procesar todas las observaciones juntas
        if observations:
            y = np.array(observations)
            H = np.array(H_matrices)
            R = np.diag(R_matrices)
            
            # Cálculo estándar de Kalman para múltiples observaciones
            innovation = y - H @ state_pred
            S = H @ P_pred @ H.T + R
            K = P_pred @ H.T @ np.linalg.inv(S)
            
            # Actualización
            state_corr = state_pred + K @ innovation
            P_corr = (np.eye(2) - K @ H) @ P_pred
            
            # Guardar métricas (solo las primeras 4 observaciones máximo)
            if len(observations) >= 1:  # Mid A
                innovations_mid_A[t] = innovation[0]
                gains_mid_A[t] = K[0, 0]
            if len(observations) >= 2:  # Mid B
                innovations_mid_B[t] = innovation[1]
                gains_mid_B[t] = K[1, 1]
            if len(observations) >= 3:  # Trade A
                innovations_trade_A[t] = innovation[2]
                gains_trade_A[t] = K[0, 2] if H.shape[0] > 2 else 0
            if len(observations) >= 4:  # Trade B
                innovations_trade_B[t] = innovation[3]
                gains_trade_B[t] = K[1, 3] if H.shape[0] > 3 else 0
        
        # Guardar resultados
        state_estimates[t] = state_corr
        state_variances[t] = P_corr
    
    # Extraer resultados por activo
    estimates_A = state_estimates[:, 0]
    estimates_B = state_estimates[:, 1]
    variances_A = state_variances[:, 0, 0]
    variances_B = state_variances[:, 1, 1]
    covariances_AB = state_variances[:, 0, 1]
    
    return {
        'estimates_A': estimates_A,
        'estimates_B': estimates_B,
        'variances_A': variances_A,
        'variances_B': variances_B,
        'covariances_AB': covariances_AB,
        'innovations_mid_A': innovations_mid_A,
        'innovations_mid_B': innovations_mid_B,
        'innovations_trade_A': innovations_trade_A,
        'innovations_trade_B': innovations_trade_B,
        'gains_mid_A': gains_mid_A,
        'gains_mid_B': gains_mid_B,
        'gains_trade_A': gains_trade_A,
        'gains_trade_B': gains_trade_B
    }

# Configurar parámetros optimizados para el filtro multiasset
MULTIASSET_KALMAN_PARAMS = {
    'Q_A': 0.01,  # Varianza proceso activo A (aumentada para spreads grandes)
    'Q_B': 0.01 * 0.8,  # Varianza proceso activo B (ligeramente menor)
    'R_mid_A': 4.0,  # Ruido mid prices activo A (spreads grandes)
    'R_mid_B': 2.0,  # Ruido mid prices activo B (spreads grandes pero menores)
    'R_trade_base': 0.5,  # Ruido base trades (ajustado por volumen)
    'initial_price_A': 100.0,
    'initial_price_B': 102.0,
    'initial_variance': 1.0
}

print("🔄 Ejecutando Filtro de Kalman Multiasset...")

# Ejecutar filtro multiasset
multiasset_results = kalman_filter_multiasset(
    market_data_A, trades_data_A,
    market_data_B, trades_data_B,
    MULTIASSET_KALMAN_PARAMS
)

print("✅ Filtro Multiasset completado")
print(f"📊 Estimaciones generadas para {len(multiasset_results['estimates_A'])} períodos")
print(f"🔗 Activo A - Varianza inicial: {multiasset_results['variances_A'][0]:.6f}")
print(f"🔗 Activo A - Varianza final: {multiasset_results['variances_A'][-1]:.6f}")
print(f"🔗 Activo B - Varianza inicial: {multiasset_results['variances_B'][0]:.6f}")  
print(f"🔗 Activo B - Varianza final: {multiasset_results['variances_B'][-1]:.6f}")
print(f"🎯 Covarianza final A-B: {multiasset_results['covariances_AB'][-1]:.6f}")

# Calcular errores
error_multiasset_A = np.abs(multiasset_results['estimates_A'] - true_prices)
error_multiasset_B = np.abs(multiasset_results['estimates_B'] - true_prices_B)

print(f"📈 MAE Activo A: ${np.mean(error_multiasset_A):.4f}")
print(f"📉 MAE Activo B: ${np.mean(error_multiasset_B):.4f}")

🔄 Ejecutando Filtro de Kalman Multiasset...
✅ Filtro Multiasset completado
📊 Estimaciones generadas para 300 períodos
🔗 Activo A - Varianza inicial: 1.000000
🔗 Activo A - Varianza final: 0.074831
🔗 Activo B - Varianza inicial: 1.000000
🔗 Activo B - Varianza final: 0.046241
🎯 Covarianza final A-B: 0.047909
📈 MAE Activo A: $0.2365
📉 MAE Activo B: $0.2740


## 3.5 Comparación: Filtros Independientes vs Multiasset

Para demostrar la ventaja del filtro multiasset, lo compararemos contra filtros de Kalman independientes que no aprovechan la correlación entre activos.

In [19]:
# Ejecutar filtros independientes para comparación
print("🔄 Ejecutando filtros independientes para comparación...")

# Parámetros para filtros individuales (basados en funciones anteriores)
independent_params_A = {
    'Q': MULTIASSET_KALMAN_PARAMS['Q_A'],
    'R_mid': MULTIASSET_KALMAN_PARAMS['R_mid_A'],
    'R_trade_base': MULTIASSET_KALMAN_PARAMS['R_trade_base'],
    'initial_price': MULTIASSET_KALMAN_PARAMS['initial_price_A'],
    'initial_variance': MULTIASSET_KALMAN_PARAMS['initial_variance']
}

independent_params_B = {
    'Q': MULTIASSET_KALMAN_PARAMS['Q_B'],
    'R_mid': MULTIASSET_KALMAN_PARAMS['R_mid_B'],
    'R_trade_base': MULTIASSET_KALMAN_PARAMS['R_trade_base'],
    'initial_price': MULTIASSET_KALMAN_PARAMS['initial_price_B'],
    'initial_variance': MULTIASSET_KALMAN_PARAMS['initial_variance']
}

# Filtro independiente para activo A
independent_results_A = kalman_filter_dual_observations(
    market_data_A['mid_price'].values, trades_data_A, 
    Q=independent_params_A['Q'],
    R_mid=independent_params_A['R_mid'],
    R_trade_base=independent_params_A['R_trade_base'],
    initial_price=independent_params_A['initial_price'],
    initial_variance=independent_params_A['initial_variance']
)

# Filtro independiente para activo B  
independent_results_B = kalman_filter_dual_observations(
    market_data_B['mid_price'].values, trades_data_B,
    Q=independent_params_B['Q'],
    R_mid=independent_params_B['R_mid'],
    R_trade_base=independent_params_B['R_trade_base'],
    initial_price=independent_params_B['initial_price'],
    initial_variance=independent_params_B['initial_variance']
)

print("✅ Filtros independientes completados")

# Calcular errores para comparación
error_independent_A = np.abs(independent_results_A['estimates'] - true_prices)
error_independent_B = np.abs(independent_results_B['estimates'] - true_prices_B)

print("\\n📊 COMPARACIÓN DE ERRORES:")
print("🔗 ACTIVO A:")
print(f"   • Filtro Independiente MAE: ${np.mean(error_independent_A):.4f}")
print(f"   • Filtro Multiasset MAE:    ${np.mean(error_multiasset_A):.4f}")
improvement_A = (np.mean(error_independent_A) - np.mean(error_multiasset_A)) / np.mean(error_independent_A) * 100
print(f"   • Mejora Multiasset: {improvement_A:+.1f}%")

print("🔗 ACTIVO B:")
print(f"   • Filtro Independiente MAE: ${np.mean(error_independent_B):.4f}")
print(f"   • Filtro Multiasset MAE:    ${np.mean(error_multiasset_B):.4f}")
improvement_B = (np.mean(error_independent_B) - np.mean(error_multiasset_B)) / np.mean(error_independent_B) * 100
print(f"   • Mejora Multiasset: {improvement_B:+.1f}%")

# Análisis en períodos sin trades
periods_no_trades_A = set(range(300)) - set(trade_times_A)
periods_no_trades_B = set(range(300)) - set(trade_times_B)
periods_no_trades_either = periods_no_trades_A & periods_no_trades_B

if len(periods_no_trades_either) > 0:
    print(f"\\n🔍 ANÁLISIS EN PERÍODOS SIN TRADES ({len(periods_no_trades_either)} períodos):")
    
    no_trades_indices = list(periods_no_trades_either)
    
    error_indep_A_no_trades = np.mean(error_independent_A[no_trades_indices])
    error_multi_A_no_trades = np.mean(error_multiasset_A[no_trades_indices])
    
    error_indep_B_no_trades = np.mean(error_independent_B[no_trades_indices])
    error_multi_B_no_trades = np.mean(error_multiasset_B[no_trades_indices])
    
    print(f"📈 Activo A (sin trades propios):")
    print(f"   • Independiente: ${error_indep_A_no_trades:.4f}")
    print(f"   • Multiasset:    ${error_multi_A_no_trades:.4f}")
    improvement_A_no_trades = (error_indep_A_no_trades - error_multi_A_no_trades) / error_indep_A_no_trades * 100
    print(f"   • Mejora: {improvement_A_no_trades:+.1f}%")
    
    print(f"📉 Activo B (sin trades propios):")
    print(f"   • Independiente: ${error_indep_B_no_trades:.4f}")
    print(f"   • Multiasset:    ${error_multi_B_no_trades:.4f}")
    improvement_B_no_trades = (error_indep_B_no_trades - error_multi_B_no_trades) / error_indep_B_no_trades * 100
    print(f"   • Mejora: {improvement_B_no_trades:+.1f}%")

🔄 Ejecutando filtros independientes para comparación...
✅ Filtros independientes completados
\n📊 COMPARACIÓN DE ERRORES:
🔗 ACTIVO A:
   • Filtro Independiente MAE: $0.6054
   • Filtro Multiasset MAE:    $0.2365
   • Mejora Multiasset: +60.9%
🔗 ACTIVO B:
   • Filtro Independiente MAE: $0.2775
   • Filtro Multiasset MAE:    $0.2740
   • Mejora Multiasset: +1.2%
\n🔍 ANÁLISIS EN PERÍODOS SIN TRADES (49 períodos):
📈 Activo A (sin trades propios):
   • Independiente: $0.5428
   • Multiasset:    $0.1996
   • Mejora: +63.2%
📉 Activo B (sin trades propios):
   • Independiente: $0.2481
   • Multiasset:    $0.2327
   • Mejora: +6.2%


In [20]:
# Crear visualización multiasset CORREGIDA con títulos precisos
fig_multiasset = make_subplots(
    rows=5, cols=2,
    subplot_titles=[
        'Activo A: Precios y Estimaciones', 'Activo B: Precios y Estimaciones',
        'Activo A: Errores Absolutos vs Spread (bps)', 'Activo B: Errores Absolutos vs Spread (bps)', 
        'Activo A: Volúmenes de Trades', 'Activo B: Volúmenes de Trades',
        'Ganancias de Kalman Mid Prices', 'Incertidumbre: Varianzas Posteriores',
        'Covarianza Cruzada A-B', 'Mejora Relativa Multiasset (%)'
    ],
    specs=[[{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": True}, {"secondary_y": True}],
           [{"type": "bar"}, {"type": "bar"}],
           [{"secondary_y": False}, {"secondary_y": False}],
           [{"secondary_y": False}, {"secondary_y": False}]],
    vertical_spacing=0.06,
    horizontal_spacing=0.08
)

time_axis = np.arange(300)

# COLORES ESTÁNDAR
COLOR_BUY = 'lime'           # Buy trades: verde
COLOR_SELL = 'red'           # Sell trades: rojo  
COLOR_BID = 'green'          # Bid: verde
COLOR_ASK = 'red'            # Ask: rojo
COLOR_MID = 'cyan'           # Mid price: cian discontinuo
COLOR_KALMAN_DUAL = 'lime'   # Kalman Dual (bid ask y trades): Lime
COLOR_KALMAN_MULTI = 'blue'  # Kalman Dual multiasset: Azul

# =============================================================================
# FILA 1: PRECIOS Y ESTIMACIONES ACTIVO A
# =============================================================================

# Precio verdadero
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=true_prices, name='Precio Verdadero A', 
               line=dict(color='blue', width=2), showlegend=True),
    row=1, col=1
)

# Bid/Ask con relleno
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=market_data_A['ask_price'], name='Ask A', 
               line=dict(color=COLOR_ASK, width=1), showlegend=True),
    row=1, col=1
)
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=market_data_A['bid_price'], name='Bid A', 
               line=dict(color=COLOR_BID, width=1), fill='tonexty', 
               fillcolor='rgba(150,150,150,0.2)', showlegend=True),
    row=1, col=1
)

# Mid price (cian discontinuo)
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=market_data_A['mid_price'], name='Mid Price A', 
               line=dict(color=COLOR_MID, width=1, dash='dash'), showlegend=True),
    row=1, col=1
)

# Trades A
if len(trades_data_A) > 0:
    buys_A = trades_data_A[trades_data_A['side'] == 1]
    sells_A = trades_data_A[trades_data_A['side'] == -1]
    
    if len(buys_A) > 0:
        fig_multiasset.add_trace(
            go.Scatter(x=buys_A['time'], y=buys_A['price'], mode='markers',
                       name='Buy Trades A', marker=dict(color=COLOR_BUY, size=8, symbol='triangle-up'),
                       showlegend=True),
            row=1, col=1
        )
    
    if len(sells_A) > 0:
        fig_multiasset.add_trace(
            go.Scatter(x=sells_A['time'], y=sells_A['price'], mode='markers',
                       name='Sell Trades A', marker=dict(color=COLOR_SELL, size=8, symbol='triangle-down'),
                       showlegend=True),
            row=1, col=1
        )

# Estimaciones Kalman
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=independent_results_A['estimates'], name='Kalman Dual A', 
               line=dict(color=COLOR_KALMAN_DUAL, width=2), showlegend=True),
    row=1, col=1
)
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=multiasset_results['estimates_A'], name='Kalman Multiasset A', 
               line=dict(color=COLOR_KALMAN_MULTI, width=2), showlegend=True),
    row=1, col=1
)

# =============================================================================
# FILA 1: PRECIOS Y ESTIMACIONES ACTIVO B
# =============================================================================

# Precio verdadero B
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=true_prices_B, name='Precio Verdadero B', 
               line=dict(color='darkgreen', width=2), showlegend=False),
    row=1, col=2
)

# Bid/Ask B
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=market_data_B['ask_price'], name='Ask B', 
               line=dict(color=COLOR_ASK, width=1), showlegend=False),
    row=1, col=2
)
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=market_data_B['bid_price'], name='Bid B', 
               line=dict(color=COLOR_BID, width=1), fill='tonexty', 
               fillcolor='rgba(150,150,150,0.2)', showlegend=False),
    row=1, col=2
)

# Mid price B (cian discontinuo)
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=market_data_B['mid_price'], name='Mid Price B', 
               line=dict(color=COLOR_MID, width=1, dash='dash'), showlegend=False),
    row=1, col=2
)

# Trades B (muestra reducida para claridad)
if len(trades_data_B) > 0:
    trades_B_sample = trades_data_B.iloc[::3]  # Cada 3er trade
    buys_B = trades_B_sample[trades_B_sample['side'] == 1]
    sells_B = trades_B_sample[trades_B_sample['side'] == -1]
    
    if len(buys_B) > 0:
        fig_multiasset.add_trace(
            go.Scatter(x=buys_B['time'], y=buys_B['price'], mode='markers',
                       name='Buy Trades B', marker=dict(color=COLOR_BUY, size=6, symbol='triangle-up'),
                       showlegend=False),
            row=1, col=2
        )
    
    if len(sells_B) > 0:
        fig_multiasset.add_trace(
            go.Scatter(x=sells_B['time'], y=sells_B['price'], mode='markers',
                       name='Sell Trades B', marker=dict(color=COLOR_SELL, size=6, symbol='triangle-down'),
                       showlegend=False),
            row=1, col=2
        )

# Estimaciones Kalman B
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=independent_results_B['estimates'], name='Kalman Dual B', 
               line=dict(color=COLOR_KALMAN_DUAL, width=2), showlegend=False),
    row=1, col=2
)
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=multiasset_results['estimates_B'], name='Kalman Multiasset B', 
               line=dict(color=COLOR_KALMAN_MULTI, width=2), showlegend=False),
    row=1, col=2
)

# =============================================================================
# FILA 2: ERRORES VS SPREAD
# =============================================================================

# Errores A
error_independent_A = np.abs(independent_results_A['estimates'] - true_prices)
error_multiasset_A = np.abs(multiasset_results['estimates_A'] - true_prices)

fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=error_independent_A, name='Error Dual A', 
               line=dict(color=COLOR_KALMAN_DUAL, width=1), showlegend=False),
    row=2, col=1
)
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=error_multiasset_A, name='Error Multiasset A', 
               line=dict(color=COLOR_KALMAN_MULTI, width=1), showlegend=False),
    row=2, col=1
)

# Spread A en eje secundario
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=market_data_A['spread_bps'], name='Spread A (bps)', 
               line=dict(color='orange', width=1), yaxis='y2', showlegend=False),
    row=2, col=1, secondary_y=True
)

# Errores B
error_independent_B = np.abs(independent_results_B['estimates'] - true_prices_B)  
error_multiasset_B = np.abs(multiasset_results['estimates_B'] - true_prices_B)

fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=error_independent_B, name='Error Dual B', 
               line=dict(color=COLOR_KALMAN_DUAL, width=1), showlegend=False),
    row=2, col=2
)
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=error_multiasset_B, name='Error Multiasset B', 
               line=dict(color=COLOR_KALMAN_MULTI, width=1), showlegend=False),
    row=2, col=2
)

# Spread B en eje secundario
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=market_data_B['spread_bps'], name='Spread B (bps)', 
               line=dict(color='orange', width=1), yaxis='y2', showlegend=False),
    row=2, col=2, secondary_y=True
)

# =============================================================================
# FILA 3: VOLÚMENES DE TRADES
# =============================================================================

# Volúmenes A (barras cuando hay trades)
volumes_by_time_A = np.zeros(300)
trade_colors_A = []
for i in range(300):
    if i in trade_times_A:
        idx = np.where(trade_times_A == i)[0][0]
        volumes_by_time_A[i] = trade_volumes_A[idx]
        trade_colors_A.append(COLOR_BUY if trade_sides_A[idx] == 1 else COLOR_SELL)
    else:
        trade_colors_A.append('rgba(100,100,100,0.1)')

fig_multiasset.add_trace(
    go.Bar(x=time_axis, y=volumes_by_time_A, name='Volumen A',
           marker=dict(color=trade_colors_A), showlegend=False),
    row=3, col=1
)

# Volúmenes B (barras cuando hay trades)
volumes_by_time_B = np.zeros(300)
trade_colors_B = []
for i in range(300):
    if i in trade_times_B:
        idx = np.where(trade_times_B == i)[0][0]
        volumes_by_time_B[i] = trade_volumes_B[idx]
        trade_colors_B.append(COLOR_BUY if trade_sides_B[idx] == 1 else COLOR_SELL)
    else:
        trade_colors_B.append('rgba(100,100,100,0.1)')

fig_multiasset.add_trace(
    go.Bar(x=time_axis, y=volumes_by_time_B, name='Volumen B',
           marker=dict(color=trade_colors_B), showlegend=False),
    row=3, col=2
)

# =============================================================================
# FILA 4: GANANCIAS DE KALMAN
# =============================================================================

# Ganancias para mid prices (siempre disponibles)
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=multiasset_results['gains_mid_A'], name='Ganancia Mid A', 
               line=dict(color='cyan', width=1), showlegend=False),
    row=4, col=1
)

fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=multiasset_results['gains_mid_B'], name='Ganancia Mid B', 
               line=dict(color='lightgreen', width=1), showlegend=False),
    row=4, col=2
)

# Ganancias para trades (solo cuando hay trades)
if len(trade_times_A) > 0:
    gains_trades_A_x = []
    gains_trades_A_y = []
    for i, gain in enumerate(multiasset_results['gains_trade_A']):
        if gain > 0:
            gains_trades_A_x.append(i)
            gains_trades_A_y.append(gain)
    
    if len(gains_trades_A_x) > 0:
        fig_multiasset.add_trace(
            go.Scatter(x=gains_trades_A_x, y=gains_trades_A_y, mode='markers',
                       name='Ganancia Trades A', marker=dict(color='red', size=6), showlegend=False),
            row=4, col=1
        )

if len(trade_times_B) > 0:
    gains_trades_B_x = []
    gains_trades_B_y = []
    for i, gain in enumerate(multiasset_results['gains_trade_B']):
        if gain > 0:
            gains_trades_B_x.append(i)
            gains_trades_B_y.append(gain)
    
    if len(gains_trades_B_x) > 0:
        fig_multiasset.add_trace(
            go.Scatter(x=gains_trades_B_x, y=gains_trades_B_y, mode='markers',
                       name='Ganancia Trades B', marker=dict(color='red', size=4), showlegend=False),
            row=4, col=2
        )

# =============================================================================
# FILA 5: CORREGIDA - SEPARACIÓN COVARIANZA VS MEJORA RELATIVA
# =============================================================================

# SUBPLOT IZQUIERDO: Varianzas + Covarianza (coherente con incertidumbre)
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=multiasset_results['variances_A'], name='Varianza A', 
               line=dict(color=COLOR_KALMAN_MULTI, width=2), showlegend=False),
    row=5, col=1
)
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=multiasset_results['variances_B'], name='Varianza B', 
               line=dict(color='darkgreen', width=2), showlegend=False),
    row=5, col=1
)
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=multiasset_results['covariances_AB'], name='Covarianza A-B', 
               line=dict(color='purple', width=2), showlegend=False),
    row=5, col=1
)

# SUBPLOT DERECHO: SOLO mejora relativa (CORREGIDA)
# Calcular mejora relativa con protección contra división por cero y valores extremos
def safe_improvement(error_ind, error_multi):
    """Calcula mejora relativa protegida contra valores extremos"""
    improvement = np.zeros_like(error_ind)
    for i in range(len(error_ind)):
        if error_ind[i] > 1e-6:  # Evitar división por valores muy pequeños
            raw_improvement = (error_ind[i] - error_multi[i]) / error_ind[i] * 100
            # Limitar valores extremos a ±200%
            improvement[i] = np.clip(raw_improvement, -200, 200)
        else:
            improvement[i] = 0
    return improvement

improvement_A_by_period = safe_improvement(error_independent_A, error_multiasset_A)
improvement_B_by_period = safe_improvement(error_independent_B, error_multiasset_B)

fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=improvement_A_by_period, name='Mejora A (%)', 
               line=dict(color=COLOR_KALMAN_MULTI, width=1), showlegend=False),
    row=5, col=2
)
fig_multiasset.add_trace(
    go.Scatter(x=time_axis, y=improvement_B_by_period, name='Mejora B (%)', 
               line=dict(color='darkgreen', width=1), showlegend=False),
    row=5, col=2
)

# Línea de referencia en 0 para mejoras
fig_multiasset.add_hline(y=0, line_dash="dash", line_color="gray", row=5, col=2)

# Configuración del layout CORREGIDA
fig_multiasset.update_layout(
    title={'text': '🔗 Filtro de Kalman Multiasset - Análisis Completo CORREGIDO', 
           'font': {'color': 'white', 'size': 18}},
    template='plotly_dark',
    height=1400,
    showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Configurar títulos de ejes Y CORRECTOS
fig_multiasset.update_yaxes(title_text="Precio ($)", row=1, col=1)
fig_multiasset.update_yaxes(title_text="Precio ($)", row=1, col=2)
fig_multiasset.update_yaxes(title_text="Error Absoluto ($)", row=2, col=1)
fig_multiasset.update_yaxes(title_text="Error Absoluto ($)", row=2, col=2)
fig_multiasset.update_yaxes(title_text="Spread (bps)", row=2, col=1, secondary_y=True)
fig_multiasset.update_yaxes(title_text="Spread (bps)", row=2, col=2, secondary_y=True)
fig_multiasset.update_yaxes(title_text="Volumen", row=3, col=1)
fig_multiasset.update_yaxes(title_text="Volumen", row=3, col=2)
fig_multiasset.update_yaxes(title_text="Ganancia Kalman (0-1)", row=4, col=1)
fig_multiasset.update_yaxes(title_text="Ganancia Kalman (0-1)", row=4, col=2)
fig_multiasset.update_yaxes(title_text="Varianza + Covarianza", row=5, col=1)
fig_multiasset.update_yaxes(title_text="Mejora Relativa (%)", row=5, col=2)

# Configurar ejes X con rangos y ticks consistentes
for i in range(1, 6):
    for j in range(1, 3):
        fig_multiasset.update_xaxes(
            title_text="Tiempo (períodos)", 
            range=[-0.5, 299.5], 
            dtick=50, 
            row=i, col=j
        )

fig_multiasset.show()

print("🎯 VISUALIZACIÓN MULTIASSET CORREGIDA Y MEJORADA")
print("\n📊 CORRECCIONES IMPLEMENTADAS:")
print("   ✅ Títulos de subplots corregidos para coincidir con contenido")
print("   ✅ Separación clara: Covarianza vs Mejora Relativa")
print("   ✅ Protección contra spikes extremos (limitado a ±200%)")
print("   ✅ Ejes Y con unidades precisas")

print(f"\n🎨 COLORES ESTÁNDAR:")
print(f"   • Buy trades: {COLOR_BUY}")
print(f"   • Sell trades: {COLOR_SELL}")  
print(f"   • Bid: {COLOR_BID}")
print(f"   • Ask: {COLOR_ASK}")
print(f"   • Mid price: {COLOR_MID} (discontinuo)")
print(f"   • Kalman Dual: {COLOR_KALMAN_DUAL}")
print(f"   • Kalman Multiasset: {COLOR_KALMAN_MULTI}")

print("\n📈 EXPLICACIÓN DE SUBPLOTS:")
print("   🔹 Fila 1: PRECIOS - Verdaderos vs Estimados")
print("   🔹 Fila 2: ERRORES - Absolutos + Spreads superpuestos") 
print("   🔹 Fila 3: VOLÚMENES - Por color (verde=compra, rojo=venta)")
print("   🔹 Fila 4: GANANCIAS - Qué tanto ajusta Kalman (0-1)")
print("   🔹 Fila 5 Izq: INCERTIDUMBRE - Varianzas + Covarianza")
print("   🔹 Fila 5 Der: MEJORA RELATIVA - % Multiasset vs Independiente")

🎯 VISUALIZACIÓN MULTIASSET CORREGIDA Y MEJORADA

📊 CORRECCIONES IMPLEMENTADAS:
   ✅ Títulos de subplots corregidos para coincidir con contenido
   ✅ Separación clara: Covarianza vs Mejora Relativa
   ✅ Protección contra spikes extremos (limitado a ±200%)
   ✅ Ejes Y con unidades precisas

🎨 COLORES ESTÁNDAR:
   • Buy trades: lime
   • Sell trades: red
   • Bid: green
   • Ask: red
   • Mid price: cyan (discontinuo)
   • Kalman Dual: lime
   • Kalman Multiasset: blue

📈 EXPLICACIÓN DE SUBPLOTS:
   🔹 Fila 1: PRECIOS - Verdaderos vs Estimados
   🔹 Fila 2: ERRORES - Absolutos + Spreads superpuestos
   🔹 Fila 3: VOLÚMENES - Por color (verde=compra, rojo=venta)
   🔹 Fila 4: GANANCIAS - Qué tanto ajusta Kalman (0-1)
   🔹 Fila 5 Izq: INCERTIDUMBRE - Varianzas + Covarianza
   🔹 Fila 5 Der: MEJORA RELATIVA - % Multiasset vs Independiente


## 3.6 Simulación Interactiva de Kalman - Primeros 15 Períodos

Esta sección muestra la **evolución paso a paso** del Filtro de Kalman Multiasset durante los primeros 15 períodos. 

### **Características de la Simulación:**
- **🎮 Interactiva**: Control deslizante para navegar período por período
- **📊 Bandas de Incertidumbre**: 2 desviaciones típicas (95.45% de confianza)
- **🔄 Fases del Filtro**: Predicción → Observación → Corrección
- **📈 Información Visual**: Trades, spreads, ganancias de Kalman en tiempo real
- **🎯 Aprendizaje**: Cómo el filtro reduce incertidumbre con cada observación

### **Interpretación:**
- **Bandas Azules**: Incertidumbre ±2σ (95.45% confianza)
- **Línea Blanca**: Precio verdadero
- **Línea Azul Sólida**: Estimación del filtro multiasset
- **Puntos Verdes/Rojos**: Trades (buy/sell) cuando disponibles
- **Ancho de Bandas**: Indica incertidumbre (se reduce con información)

In [21]:
# Instalar ipywidgets si no está disponible
try:
    import ipywidgets as widgets
    from IPython.display import display
    print("✅ ipywidgets disponible")
except ImportError:
    print("⚠️ Instalando ipywidgets...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "ipywidgets"])
    import ipywidgets as widgets
    from IPython.display import display
    print("✅ ipywidgets instalado y listo")

# Configuración para la simulación interactiva de 15 períodos
SIMULATION_PERIODS = 15

def create_interactive_kalman_simulation():
    """
    Crea simulación interactiva del Filtro de Kalman Multiasset
    mostrando la evolución paso a paso con bandas de incertidumbre de 2σ
    """
    
    def plot_kalman_step(period_end=1):
        """
        Dibuja la evolución del filtro hasta el período especificado
        con bandas de incertidumbre de 2 desviaciones típicas
        """
        # Asegurar que period_end esté en rango válido
        period_end = max(1, min(period_end, SIMULATION_PERIODS))
        
        # Extraer datos hasta el período actual
        time_subset = np.arange(period_end + 1)
        
        # Datos verdaderos
        true_A_subset = true_prices[:period_end + 1]
        true_B_subset = true_prices_B[:period_end + 1]
        
        # Estimaciones multiasset
        est_A_subset = multiasset_results['estimates_A'][:period_end + 1]
        est_B_subset = multiasset_results['estimates_B'][:period_end + 1]
        
        # Incertidumbre (2 desviaciones típicas = 95.45% confianza)
        var_A_subset = multiasset_results['variances_A'][:period_end + 1]
        var_B_subset = multiasset_results['variances_B'][:period_end + 1]
        
        # Bandas de confianza 2σ
        std_A = np.sqrt(var_A_subset)
        std_B = np.sqrt(var_B_subset)
        
        upper_A = est_A_subset + 2 * std_A
        lower_A = est_A_subset - 2 * std_A
        upper_B = est_B_subset + 2 * std_B
        lower_B = est_B_subset - 2 * std_B
        
        # Market data hasta el período actual
        bid_A_subset = market_data_A['bid_price'][:period_end + 1]
        ask_A_subset = market_data_A['ask_price'][:period_end + 1]
        mid_A_subset = market_data_A['mid_price'][:period_end + 1]
        
        bid_B_subset = market_data_B['bid_price'][:period_end + 1]
        ask_B_subset = market_data_B['ask_price'][:period_end + 1]
        mid_B_subset = market_data_B['mid_price'][:period_end + 1]
        
        # Trades hasta el período actual
        trades_A_current = trades_data_A[trades_data_A['time'] <= period_end]
        trades_B_current = trades_data_B[trades_data_B['time'] <= period_end]
        
        # Crear figura con 3 subplots
        fig = make_subplots(
            rows=3, cols=2,
            subplot_titles=[
                f'Activo A - Evolución hasta Período {period_end}',
                f'Activo B - Evolución hasta Período {period_end}',
                'Trades y Volúmenes A', 'Trades y Volúmenes B',
                'Spread e Incertidumbre A', 'Spread e Incertidumbre B'
            ],
            specs=[[{"secondary_y": False}, {"secondary_y": False}],
                   [{"type": "bar"}, {"type": "bar"}],
                   [{"secondary_y": True}, {"secondary_y": True}]],
            vertical_spacing=0.12,
            horizontal_spacing=0.1
        )
        
        # FILA 1: PRECIOS Y ESTIMACIONES CON BANDAS
        
        # ACTIVO A
        # Banda de incertidumbre 2σ (relleno azul)
        fig.add_trace(
            go.Scatter(x=time_subset, y=upper_A, name='', 
                       line=dict(color='rgba(0,0,0,0)'), showlegend=False),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(x=time_subset, y=lower_A, name='Incertidumbre ±2σ A', 
                       fill='tonexty', fillcolor='rgba(0,100,255,0.3)',
                       line=dict(color='rgba(0,0,0,0)'), showlegend=True),
            row=1, col=1
        )
        
        # Precio verdadero
        fig.add_trace(
            go.Scatter(x=time_subset, y=true_A_subset, name='Precio Verdadero A',
                       line=dict(color='white', width=3, dash='dot'), showlegend=True),
            row=1, col=1
        )
        
        # Bid/Ask spread
        fig.add_trace(
            go.Scatter(x=time_subset, y=ask_A_subset, name='Ask A',
                       line=dict(color='red', width=1), showlegend=True),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(x=time_subset, y=bid_A_subset, name='Bid A',
                       line=dict(color='green', width=1), fill='tonexty',
                       fillcolor='rgba(255,200,200,0.2)', showlegend=True),
            row=1, col=1
        )
        
        # Mid price
        fig.add_trace(
            go.Scatter(x=time_subset, y=mid_A_subset, name='Mid Price A',
                       line=dict(color='cyan', width=1, dash='dash'), showlegend=True),
            row=1, col=1
        )
        
        # Estimación Kalman
        fig.add_trace(
            go.Scatter(x=time_subset, y=est_A_subset, name='Kalman Multiasset A',
                       line=dict(color='blue', width=3), showlegend=True),
            row=1, col=1
        )
        
        # Trades A
        if len(trades_A_current) > 0:
            buys_A_current = trades_A_current[trades_A_current['side'] == 1]
            sells_A_current = trades_A_current[trades_A_current['side'] == -1]
            
            if len(buys_A_current) > 0:
                fig.add_trace(
                    go.Scatter(x=buys_A_current['time'], y=buys_A_current['price'],
                               mode='markers', name='Buy Trades A',
                               marker=dict(color='lime', size=12, symbol='triangle-up'),
                               showlegend=True),
                    row=1, col=1
                )
            
            if len(sells_A_current) > 0:
                fig.add_trace(
                    go.Scatter(x=sells_A_current['time'], y=sells_A_current['price'],
                               mode='markers', name='Sell Trades A',
                               marker=dict(color='red', size=12, symbol='triangle-down'),
                               showlegend=True),
                    row=1, col=1
                )
        
        # ACTIVO B (similar estructura)
        # Banda de incertidumbre 2σ
        fig.add_trace(
            go.Scatter(x=time_subset, y=upper_B, name='',
                       line=dict(color='rgba(0,0,0,0)'), showlegend=False),
            row=1, col=2
        )
        fig.add_trace(
            go.Scatter(x=time_subset, y=lower_B, name='Incertidumbre ±2σ B',
                       fill='tonexty', fillcolor='rgba(0,100,255,0.3)',
                       line=dict(color='rgba(0,0,0,0)'), showlegend=False),
            row=1, col=2
        )
        
        # Precio verdadero B
        fig.add_trace(
            go.Scatter(x=time_subset, y=true_B_subset, name='Precio Verdadero B',
                       line=dict(color='white', width=3, dash='dot'), showlegend=False),
            row=1, col=2
        )
        
        # Bid/Ask B
        fig.add_trace(
            go.Scatter(x=time_subset, y=ask_B_subset, name='Ask B',
                       line=dict(color='red', width=1), showlegend=False),
            row=1, col=2
        )
        fig.add_trace(
            go.Scatter(x=time_subset, y=bid_B_subset, name='Bid B',
                       line=dict(color='green', width=1), fill='tonexty',
                       fillcolor='rgba(200,255,200,0.2)', showlegend=False),
            row=1, col=2
        )
        
        # Mid price B
        fig.add_trace(
            go.Scatter(x=time_subset, y=mid_B_subset, name='Mid Price B',
                       line=dict(color='cyan', width=1, dash='dash'), showlegend=False),
            row=1, col=2
        )
        
        # Estimación Kalman B
        fig.add_trace(
            go.Scatter(x=time_subset, y=est_B_subset, name='Kalman Multiasset B',
                       line=dict(color='darkblue', width=3), showlegend=False),
            row=1, col=2
        )
        
        # Trades B
        if len(trades_B_current) > 0:
            buys_B_current = trades_B_current[trades_B_current['side'] == 1]
            sells_B_current = trades_B_current[trades_B_current['side'] == -1]
            
            if len(buys_B_current) > 0:
                fig.add_trace(
                    go.Scatter(x=buys_B_current['time'], y=buys_B_current['price'],
                               mode='markers', name='Buy Trades B',
                               marker=dict(color='lime', size=10, symbol='triangle-up'),
                               showlegend=False),
                    row=1, col=2
                )
            
            if len(sells_B_current) > 0:
                fig.add_trace(
                    go.Scatter(x=sells_B_current['time'], y=sells_B_current['price'],
                               mode='markers', name='Sell Trades B',
                               marker=dict(color='red', size=10, symbol='triangle-down'),
                               showlegend=False),
                    row=1, col=2
                )
        
        # FILA 2: VOLÚMENES DE TRADES
        
        # Volúmenes A
        volumes_A_subset = np.zeros(period_end + 1)
        colors_A_subset = []
        
        for i in range(period_end + 1):
            if i in trades_A_current['time'].values:
                trade_at_time = trades_A_current[trades_A_current['time'] == i].iloc[0]
                volumes_A_subset[i] = trade_at_time['volume']
                colors_A_subset.append('lime' if trade_at_time['side'] == 1 else 'red')
            else:
                colors_A_subset.append('rgba(100,100,100,0.1)')
        
        fig.add_trace(
            go.Bar(x=time_subset, y=volumes_A_subset, name='Volumen Trades A',
                   marker=dict(color=colors_A_subset), showlegend=False),
            row=2, col=1
        )
        
        # Volúmenes B
        volumes_B_subset = np.zeros(period_end + 1)
        colors_B_subset = []
        
        for i in range(period_end + 1):
            if i in trades_B_current['time'].values:
                trade_at_time = trades_B_current[trades_B_current['time'] == i].iloc[0]
                volumes_B_subset[i] = trade_at_time['volume']
                colors_B_subset.append('lime' if trade_at_time['side'] == 1 else 'red')
            else:
                colors_B_subset.append('rgba(100,100,100,0.1)')
        
        fig.add_trace(
            go.Bar(x=time_subset, y=volumes_B_subset, name='Volumen Trades B',
                   marker=dict(color=colors_B_subset), showlegend=False),
            row=2, col=2
        )
        
        # FILA 3: SPREADS E INCERTIDUMBRE
        
        # Spreads A (en basis points)
        spreads_A_subset = market_data_A['spread_bps'][:period_end + 1]
        fig.add_trace(
            go.Scatter(x=time_subset, y=spreads_A_subset, name='Spread A (bps)',
                       line=dict(color='orange', width=2), showlegend=False),
            row=3, col=1
        )
        
        # Incertidumbre A (desviación estándar en eje secundario)
        uncertainty_A_bps = std_A * 10000  # Convertir a basis points
        fig.add_trace(
            go.Scatter(x=time_subset, y=uncertainty_A_bps, name='Incertidumbre A (bps)',
                       line=dict(color='purple', width=2), yaxis='y2', showlegend=False),
            row=3, col=1, secondary_y=True
        )
        
        # Spreads B
        spreads_B_subset = market_data_B['spread_bps'][:period_end + 1]
        fig.add_trace(
            go.Scatter(x=time_subset, y=spreads_B_subset, name='Spread B (bps)',
                       line=dict(color='orange', width=2), showlegend=False),
            row=3, col=2
        )
        
        # Incertidumbre B (desviación estándar en eje secundario)
        uncertainty_B_bps = std_B * 10000  # Convertir a basis points
        fig.add_trace(
            go.Scatter(x=time_subset, y=uncertainty_B_bps, name='Incertidumbre B (bps)',
                       line=dict(color='purple', width=2), yaxis='y2', showlegend=False),
            row=3, col=2, secondary_y=True
        )
        
        # Layout y configuración
        fig.update_layout(
            title={
                'text': f'🎮 Simulación Interactiva Kalman Multiasset - Período {period_end}/{SIMULATION_PERIODS}<br><sub>Bandas de Incertidumbre: ±2σ (95.45% confianza)</sub>',
                'font': {'color': 'white', 'size': 16}
            },
            template='plotly_dark',
            height=900,
            showlegend=True,
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
        )
        
        # Configurar ejes
        for i in range(1, 4):
            for j in range(1, 3):
                fig.update_xaxes(
                    title_text="Período",
                    range=[-0.2, SIMULATION_PERIODS + 0.2],
                    dtick=2,
                    row=i, col=j
                )
        
        # Títulos Y específicos
        fig.update_yaxes(title_text="Precio ($)", row=1, col=1)
        fig.update_yaxes(title_text="Precio ($)", row=1, col=2)
        fig.update_yaxes(title_text="Volumen", row=2, col=1)
        fig.update_yaxes(title_text="Volumen", row=2, col=2)
        fig.update_yaxes(title_text="Spread (bps)", row=3, col=1)
        fig.update_yaxes(title_text="Spread (bps)", row=3, col=2)
        fig.update_yaxes(title_text="Incertidumbre (bps)", row=3, col=1, secondary_y=True)
        fig.update_yaxes(title_text="Incertidumbre (bps)", row=3, col=2, secondary_y=True)
        
        fig.show()
        
        # Información textual del período actual
        print(f"\\n📊 INFORMACIÓN PERÍODO {period_end}:")
        
        if period_end > 0:
            print(f"🎯 Estimación A: ${est_A_subset[period_end]:.4f} (Real: ${true_A_subset[period_end]:.4f})")
            print(f"🎯 Estimación B: ${est_B_subset[period_end]:.4f} (Real: ${true_B_subset[period_end]:.4f})")
            print(f"📏 Incertidumbre A: ±{std_A[period_end]:.4f} (±2σ = ±${2*std_A[period_end]:.4f})")
            print(f"📏 Incertidumbre B: ±{std_B[period_end]:.4f} (±2σ = ±${2*std_B[period_end]:.4f})")
            
            # Información sobre trades en este período
            trades_A_t = trades_A_current[trades_A_current['time'] == period_end]
            trades_B_t = trades_B_current[trades_B_current['time'] == period_end]
            
            if len(trades_A_t) > 0:
                trade = trades_A_t.iloc[0]
                side_str = "📈 COMPRA" if trade['side'] == 1 else "📉 VENTA"
                print(f"🔔 Trade A: {side_str} ${trade['price']:.4f} Vol:{trade['volume']}")
            else:
                print("❌ Sin trade A en este período")
                
            if len(trades_B_t) > 0:
                trade = trades_B_t.iloc[0]
                side_str = "📈 COMPRA" if trade['side'] == 1 else "📉 VENTA"
                print(f"🔔 Trade B: {side_str} ${trade['price']:.4f} Vol:{trade['volume']}")
            else:
                print("❌ Sin trade B en este período")
    
    # Crear widget interactivo
    period_slider = widgets.IntSlider(
        value=1,
        min=1,
        max=SIMULATION_PERIODS,
        step=1,
        description='Período:',
        disabled=False,
        continuous_update=False,
        orientation='horizontal',
        readout=True,
        readout_format='d',
        style={'description_width': 'initial'}
    )
    
    # Widget interactivo
    interactive_plot = widgets.interactive(plot_kalman_step, period_end=period_slider)
    
    # Mostrar controles y plot
    display(interactive_plot)
    
    print("\\n🎮 CONTROLES INTERACTIVOS:")
    print("   • Desliza el control para navegar entre períodos 1-15")
    print("   • Observa cómo cambian las bandas de incertidumbre")
    print("   • Las bandas azules muestran confianza del 95.45% (±2σ)")
    print("   • Nota cómo los trades reducen la incertidumbre")

# Ejecutar simulación interactiva
print("🚀 CREANDO SIMULACIÓN INTERACTIVA DE KALMAN MULTIASSET")
print("=" * 60)

create_interactive_kalman_simulation()

✅ ipywidgets disponible
🚀 CREANDO SIMULACIÓN INTERACTIVA DE KALMAN MULTIASSET


interactive(children=(IntSlider(value=1, continuous_update=False, description='Período:', max=15, min=1, style…

\n🎮 CONTROLES INTERACTIVOS:
   • Desliza el control para navegar entre períodos 1-15
   • Observa cómo cambian las bandas de incertidumbre
   • Las bandas azules muestran confianza del 95.45% (±2σ)
   • Nota cómo los trades reducen la incertidumbre


### 📋 Resumen de Características Implementadas

#### ✅ **Simulación Interactiva:**
- **🎮 Control deslizante**: Navega período por período (1-15)
- **⚡ Tiempo real**: Actualización instantánea al cambiar período
- **🔄 Evolución progresiva**: Muestra cómo evoluciona el filtro paso a paso

#### ✅ **Bandas de Incertidumbre - 2 Desviaciones Típicas:**
- **📊 Intervalos de confianza**: 95.45% de probabilidad (±2σ)
- **🎨 Visualización**: Bandas grises rellenas alrededor de la estimación
- **📉 Reducción progresiva**: Se observa cómo disminuye con cada observación

#### ✅ **Información Completa por Período:**
- **🎯 Estimaciones vs Real**: Comparación numérica precisa
- **📏 Métricas de incertidumbre**: Valores exactos de ±2σ
- **🔔 Información de trades**: Detalle de trades cuando ocurren
- **📈 Evolución visual**: 3 subplots coordinados

#### ✅ **Interpretación Educativa:**
- **Bandas anchas**: Alta incertidumbre (pocos datos)
- **Bandas estrechas**: Baja incertidumbre (muchos datos/información)
- **Trades**: Reducen inmediatamente la incertidumbre
- **Correlación**: Activo B ayuda a mejorar estimación de A

La simulación demuestra perfectamente cómo el **Filtro de Kalman Multiasset** aprovecha la información correlacionada para mejorar las estimaciones de precio, especialmente durante períodos con pocos trades directos.